# <span style="color:blue">DATA PREPROCESSING</span> #

## <span style="color:blue">PACKAGES USED</span> ##

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
import kagglehub
from pathlib import Path
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq
import gc
import shutil


/usr/local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## <span style="color:blue">RAW DATA ACQUISITION AND TRANSFORMATION INTO PRIMARY DATASET</span> ##

### <span style="color:blue"> checking whether primary data already exist and downloading them if they do not </span> ###

In [2]:
# ============================================================
# 01. SETTINGS
# ============================================================

kaggle_dataset = "kartik2112/fraud-detection"

data_folder = Path(
    "/projeto_tcc_2026/data/initial_dataset"
)

temporary_download_folder = (
    data_folder
    / "_temporary_kaggle_download"
)

train_file = (
    data_folder
    / "fraudTrain.csv"
)

test_file = (
    data_folder
    / "fraudTest.csv"
)


# ============================================================
# 02. ENSURE THE DATA DIRECTORY EXISTS
# ============================================================

data_folder.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 03. PREPARE A CLEAN TEMPORARY DOWNLOAD DIRECTORY
# ============================================================

if temporary_download_folder.exists():

    shutil.rmtree(
        temporary_download_folder
    )


temporary_download_folder.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 04. DEFINE REQUIRED RAW DATA FILES
# ============================================================

required_files = {
    "fraudTrain.csv": train_file,
    "fraudTest.csv": test_file
}


# ============================================================
# 05. DOWNLOAD FRESH COPIES FROM KAGGLE
# ============================================================

for file_name in required_files:

    print(
        f"\nDownloading fresh copy of {file_name}..."
    )


    kagglehub.dataset_download(
        kaggle_dataset,
        path=file_name,
        output_dir=str(
            temporary_download_folder
        )
    )


# ============================================================
# 06. VALIDATE DOWNLOADED FILES
# ============================================================

for file_name in required_files:

    downloaded_file = (
        temporary_download_folder
        / file_name
    )


    file_is_valid = (
        downloaded_file.exists()
        and
        downloaded_file.is_file()
        and
        downloaded_file.stat().st_size > 0
    )


    if not file_is_valid:

        raise FileNotFoundError(
            f"Downloaded file is missing or empty: "
            f"{downloaded_file}"
        )


print(
    "\nBoth fresh dataset files were downloaded successfully."
)


# ============================================================
# 07. REPLACE THE EXISTING RAW DATA FILES
# ============================================================

for file_name, destination_file in required_files.items():

    downloaded_file = (
        temporary_download_folder
        / file_name
    )


    shutil.copy2(
        downloaded_file,
        destination_file
    )


    print(
        f"{file_name} replaced successfully."
    )


# ============================================================
# 08. REMOVE TEMPORARY DOWNLOAD FILES
# ============================================================

if temporary_download_folder.exists():

    shutil.rmtree(
        temporary_download_folder
    )


print(
    "\nTemporary download directory removed."
)


# ============================================================
# 09. REMOVE KAGGLE DOWNLOAD ARTIFACTS
# ============================================================

complete_folder = (
    data_folder
    / ".complete"
)


if complete_folder.exists():

    shutil.rmtree(
        complete_folder
    )


    print(
        ".complete folder removed successfully."
    )

else:

    print(
        ".complete folder was not found."
    )


# ============================================================
# 10. FINAL RAW DATA VALIDATION
# ============================================================

for file_name, file_path in required_files.items():

    file_is_valid = (
        file_path.exists()
        and
        file_path.is_file()
        and
        file_path.stat().st_size > 0
    )


    if not file_is_valid:

        raise FileNotFoundError(
            f"{file_name} is missing or empty in "
            f"{data_folder}."
        )


print(
    "\nFresh raw dataset files are available."
)


# ============================================================
# 11. LOAD THE ORIGINAL DATASETS
# ============================================================

print(
    "\nLoading the refreshed original datasets..."
)


train = pd.read_csv(
    train_file
)


test = pd.read_csv(
    test_file
)


# ============================================================
# 12. DISPLAY ORIGINAL DATASET DIMENSIONS
# ============================================================

print(
    "\nORIGINAL DATASET DIMENSIONS"
)

print(
    "=" * 100
)

print(
    "fraudTrain:",
    train.shape
)

print(
    "fraudTest: ",
    test.shape
)

100%|██████████| 141M/141M [00:09<00:00, 15.1MB/s] 


Extracting zip of fraudTrain.csv...



100%|██████████| 60.5M/60.5M [00:05<00:00, 11.8MB/s]

Extracting zip of fraudTest.csv...



Both fresh dataset files were downloaded successfully.
fraudTrain.csv replaced successfully.
fraudTest.csv replaced successfully.

Temporary download directory removed.
.complete folder was not found.

Fresh raw dataset files are available.

Loading the refreshed original datasets...

ORIGINAL DATASET DIMENSIONS
fraudTrain: (1296675, 23)
fraudTest:  (555719, 23)


### <span style="color:blue"> develop the primary dataset </span> ###

In [3]:
# ============================================================
# 01. DEFINE PRIMARY DATASET PATHS
# ============================================================

primary_dataset_file = (
    data_folder
    / "dataset_primary.csv"
)

temporary_primary_dataset_file = (
    data_folder
    / "_dataset_primary.csv"
)


# ============================================================
# 02. VALIDATE SOURCE DATAFRAMES
# ============================================================

if train.empty:
    raise ValueError(
        "The Train dataset is empty."
    )

if test.empty:
    raise ValueError(
        "The Test dataset is empty."
    )


print(
    "\nSOURCE DATASET VALIDATION"
)

print(
    "=" * 100
)

print(
    "Train rows:",
    train.shape[0]
)

print(
    "Train features:",
    train.shape[1]
)

print(
    "Test rows:",
    test.shape[0]
)

print(
    "Test features:",
    test.shape[1]
)


# ============================================================
# 03. REMOVE THE ORIGINAL CSV INDEX COLUMN
# ============================================================

technical_columns_to_remove = [
    "Unnamed: 0"
]


train_columns_removed = [
    column
    for column in technical_columns_to_remove
    if column in train.columns
]


test_columns_removed = [
    column
    for column in technical_columns_to_remove
    if column in test.columns
]


train = train.drop(
    columns=train_columns_removed
)


test = test.drop(
    columns=test_columns_removed
)


print(
    "\nTECHNICAL COLUMN REMOVAL"
)

print(
    "=" * 100
)

print(
    "Columns removed from Train:",
    train_columns_removed
)

print(
    "Columns removed from Test:",
    test_columns_removed
)


# ============================================================
# 04. CHECK FOR DUPLICATED COLUMN NAMES
# ============================================================

duplicated_train_columns = (
    train.columns[
        train.columns.duplicated()
    ]
    .tolist()
)


duplicated_test_columns = (
    test.columns[
        test.columns.duplicated()
    ]
    .tolist()
)


if duplicated_train_columns:

    raise ValueError(
        "Duplicated column names detected in Train: "
        f"{duplicated_train_columns}"
    )


if duplicated_test_columns:

    raise ValueError(
        "Duplicated column names detected in Test: "
        f"{duplicated_test_columns}"
    )


# ============================================================
# 05. VALIDATE SOURCE FEATURE STRUCTURE
# ============================================================

train_columns = list(
    train.columns
)

test_columns = list(
    test.columns
)


if train_columns != test_columns:

    train_only_features = sorted(
        set(
            train_columns
        )
        -
        set(
            test_columns
        )
    )


    test_only_features = sorted(
        set(
            test_columns
        )
        -
        set(
            train_columns
        )
    )


    same_features_different_order = (
        set(
            train_columns
        )
        ==
        set(
            test_columns
        )
    )


    print(
        "\nDATASET STRUCTURE WARNING"
    )

    print(
        "=" * 100
    )


    if same_features_different_order:

        print(
            "Train and Test contain the same features "
            "but in a different column order."
        )

    else:

        print(
            "Train and Test contain different features."
        )


    print(
        "\nFeatures only in Train:"
    )

    print(
        train_only_features
    )


    print(
        "\nFeatures only in Test:"
    )

    print(
        test_only_features
    )


    raise ValueError(
        "Train and Test do not have identical "
        "feature structures."
    )


print(
    "\nTrain and Test have identical feature structures."
)


# ============================================================
# 06. STORE SOURCE DATASET DIMENSIONS
# ============================================================

train_rows = int(
    train.shape[0]
)

test_rows = int(
    test.shape[0]
)

source_feature_count = int(
    train.shape[1]
)


expected_primary_rows = (
    train_rows
    +
    test_rows
)


# ============================================================
# 07. MERGE TRAIN AND TEST
# ============================================================

print(
    "\nMerging Train and Test datasets..."
)


primary_dataset = pd.concat(
    [
        train,
        test
    ],
    axis=0,
    ignore_index=True
)


# ============================================================
# 08. VALIDATE DATASET MERGE
# ============================================================

actual_primary_rows = int(
    primary_dataset.shape[0]
)

actual_primary_features = int(
    primary_dataset.shape[1]
)


if actual_primary_rows != expected_primary_rows:

    raise ValueError(
        "Primary dataset row count does not match "
        "Train rows plus Test rows."
    )


if actual_primary_features != source_feature_count:

    raise ValueError(
        "Primary dataset feature count changed "
        "unexpectedly during concatenation."
    )


if list(
    primary_dataset.columns
) != train_columns:

    raise ValueError(
        "Primary dataset column structure changed "
        "unexpectedly during concatenation."
    )


print(
    "Dataset merge validated successfully."
)


# ============================================================
# 09. CREATE THE UNIQUE OBSERVATION IDENTIFIER
# ============================================================

if "NID" in primary_dataset.columns:

    raise ValueError(
        "NID already exists before identifier creation."
    )


primary_dataset.insert(
    0,
    "NID",
    np.arange(
        1,
        len(primary_dataset) + 1,
        dtype=np.int64
    )
)


# ============================================================
# 10. VALIDATE THE UNIQUE OBSERVATION IDENTIFIER
# ============================================================

first_nid = int(
    primary_dataset[
        "NID"
    ].iloc[0]
)


last_nid = int(
    primary_dataset[
        "NID"
    ].iloc[-1]
)


unique_nids = int(
    primary_dataset[
        "NID"
    ].nunique()
)


duplicated_nids = int(
    primary_dataset[
        "NID"
    ]
    .duplicated()
    .sum()
)


missing_nids = int(
    primary_dataset[
        "NID"
    ]
    .isna()
    .sum()
)


expected_last_nid = int(
    len(
        primary_dataset
    )
)


if first_nid != 1:

    raise ValueError(
        "NID does not start at 1."
    )


if last_nid != expected_last_nid:

    raise ValueError(
        "The final NID does not match "
        "the number of observations."
    )


if unique_nids != expected_last_nid:

    raise ValueError(
        "NID is not unique for every observation."
    )


if duplicated_nids != 0:

    raise ValueError(
        "Duplicated NID values were detected."
    )


if missing_nids != 0:

    raise ValueError(
        "Missing NID values were detected."
    )


print(
    "\nNID VALIDATION"
)

print(
    "=" * 100
)

print(
    "First NID:",
    first_nid
)

print(
    "Last NID:",
    last_nid
)

print(
    "Number of unique NIDs:",
    unique_nids
)

print(
    "Number of duplicated NIDs:",
    duplicated_nids
)

print(
    "Number of missing NIDs:",
    missing_nids
)


# ============================================================
# 11. DISPLAY PRIMARY DATASET DIMENSIONS
# ============================================================

print(
    "\nPRIMARY DATASET DIMENSIONS"
)

print(
    "=" * 100
)

print(
    "Train rows:",
    train_rows
)

print(
    "Test rows:",
    test_rows
)

print(
    "Expected primary rows:",
    expected_primary_rows
)

print(
    "Primary dataset rows:",
    primary_dataset.shape[0]
)

print(
    "Primary dataset features:",
    primary_dataset.shape[1]
)


# ============================================================
# 12. REMOVE PREVIOUS TEMPORARY OUTPUT
# ============================================================

if temporary_primary_dataset_file.exists():

    temporary_primary_dataset_file.unlink()

    print(
        "\nPrevious temporary primary dataset removed."
    )


# ============================================================
# 13. SAVE THE REFRESHED PRIMARY DATASET TEMPORARILY
# ============================================================

print(
    "\nSaving refreshed primary dataset..."
)


primary_dataset.to_csv(
    temporary_primary_dataset_file,
    index=False
)


# ============================================================
# 14. VALIDATE THE TEMPORARY OUTPUT FILE
# ============================================================

temporary_file_is_valid = (
    temporary_primary_dataset_file.exists()
    and
    temporary_primary_dataset_file.is_file()
    and
    temporary_primary_dataset_file.stat().st_size > 0
)


if not temporary_file_is_valid:

    raise FileNotFoundError(
        "The temporary primary dataset was not "
        "created correctly."
    )


print(
    "Temporary primary dataset validated successfully."
)


# ============================================================
# 15. REPLACE THE PREVIOUS PRIMARY DATASET
# ============================================================

primary_dataset_already_existed = (
    primary_dataset_file.exists()
)


temporary_primary_dataset_file.replace(
    primary_dataset_file
)


# ============================================================
# 16. VALIDATE THE FINAL OUTPUT FILE
# ============================================================

primary_file_is_valid = (
    primary_dataset_file.exists()
    and
    primary_dataset_file.is_file()
    and
    primary_dataset_file.stat().st_size > 0
)


if not primary_file_is_valid:

    raise FileNotFoundError(
        "dataset_primary.csv was not created correctly."
    )


# ============================================================
# 17. DISPLAY FINAL FILE INFORMATION
# ============================================================

print(
    "\nPRIMARY DATASET FILE"
)

print(
    "=" * 100
)


if primary_dataset_already_existed:

    print(
        "Previous dataset_primary.csv overwritten successfully."
    )

else:

    print(
        "dataset_primary.csv created successfully."
    )


print(
    "Location:",
    primary_dataset_file
)


print(
    "File size:",
    f"{primary_dataset_file.stat().st_size / (1024 ** 2):.2f} MB"
)


# ============================================================
# 18. RELEASE SOURCE DATAFRAMES
# ============================================================

del train
del test

gc.collect()


print(
    "\nSource Train and Test DataFrames released from memory."
)


# ============================================================
# 19. FINAL CONFIRMATION
# ============================================================

print(
    "\nPRIMARY DATASET CREATION COMPLETED"
)

print(
    "=" * 100
)

print(
    "Observations:",
    primary_dataset.shape[0]
)

print(
    "Features:",
    primary_dataset.shape[1]
)

print(
    "Output:",
    primary_dataset_file
)


SOURCE DATASET VALIDATION
Train rows: 1296675
Train features: 23
Test rows: 555719
Test features: 23

TECHNICAL COLUMN REMOVAL
Columns removed from Train: ['Unnamed: 0']
Columns removed from Test: ['Unnamed: 0']

Train and Test have identical feature structures.

Merging Train and Test datasets...
Dataset merge validated successfully.

NID VALIDATION
First NID: 1
Last NID: 1852394
Number of unique NIDs: 1852394
Number of duplicated NIDs: 0
Number of missing NIDs: 0

PRIMARY DATASET DIMENSIONS
Train rows: 1296675
Test rows: 555719
Expected primary rows: 1852394
Primary dataset rows: 1852394
Primary dataset features: 23

Saving refreshed primary dataset...
Temporary primary dataset validated successfully.

PRIMARY DATASET FILE
Previous dataset_primary.csv overwritten successfully.
Location: /projeto_tcc_2026/data/initial_dataset/dataset_primary.csv
File size: 472.78 MB

Source Train and Test DataFrames released from memory.

PRIMARY DATASET CREATION COMPLETED
Observations: 1852394
Featu

## <span style="color:blue">PRIMARY DATASET TRANSFORMATION AND MEDIUM DATASET CREATION</span>

In [4]:
# ============================================================
# 01. SETTINGS
# ============================================================

project_root = Path(
    "/projeto_tcc_2026"
)

primary_dataset_file = (
    project_root
    / "data"
    / "initial_dataset"
    / "dataset_primary.csv"
)

medium_data_folder = (
    project_root
    / "data"
    / "medium_dataset"
)

medium_dataset_file = (
    medium_data_folder
    / "dataset_medium.csv"
)

temporary_medium_dataset_file = (
    medium_data_folder
    / "_dataset_medium.csv"
)

reference_date = pd.Timestamp(
    "2026-09-03"
)


# ============================================================
# 02. ENSURE THE MEDIUM DATASET DIRECTORY EXISTS
# ============================================================

medium_data_folder.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 03. VALIDATE THE PRIMARY DATASET FILE
# ============================================================

primary_file_is_valid = (
    primary_dataset_file.exists()
    and
    primary_dataset_file.is_file()
    and
    primary_dataset_file.stat().st_size > 0
)


if not primary_file_is_valid:

    raise FileNotFoundError(
        "dataset_primary.csv is missing or empty: "
        f"{primary_dataset_file}"
    )


print(
    "\nPrimary dataset file validated successfully."
)


# ============================================================
# 04. LOAD THE PRIMARY DATASET
# ============================================================

print(
    "\nLoading the primary dataset..."
)


df = pd.read_csv(
    primary_dataset_file
)


if df.empty:

    raise ValueError(
        "The primary dataset is empty."
    )


initial_rows = int(
    df.shape[0]
)

initial_features = int(
    df.shape[1]
)


print(
    "\nPRIMARY DATASET DIMENSIONS"
)

print(
    "=" * 100
)

print(
    "Rows:",
    initial_rows
)

print(
    "Features:",
    initial_features
)


# ============================================================
# 05. CHECK FOR DUPLICATED COLUMN NAMES
# ============================================================

duplicated_columns = (
    df.columns[
        df.columns.duplicated()
    ]
    .tolist()
)


if duplicated_columns:

    raise ValueError(
        "Duplicated column names detected: "
        f"{duplicated_columns}"
    )


# ============================================================
# 06. VALIDATE REQUIRED PRIMARY DATASET FEATURES
# ============================================================

required_primary_features = [
    "NID",
    "trans_date_trans_time",
    "cc_num",
    "merchant",
    "category",
    "amt",
    "first",
    "last",
    "gender",
    "street",
    "city",
    "state",
    "zip",
    "lat",
    "long",
    "city_pop",
    "job",
    "dob",
    "trans_num",
    "unix_time",
    "merch_lat",
    "merch_long"
]


missing_primary_features = [
    feature
    for feature in required_primary_features
    if feature not in df.columns
]


if missing_primary_features:

    raise KeyError(
        "Missing required primary dataset features: "
        + ", ".join(
            missing_primary_features
        )
    )


# ============================================================
# 07. IDENTIFY AND VALIDATE THE TARGET FEATURE
# ============================================================

target_candidates = [
    feature
    for feature in [
        "is_fraud",
        "is_fraude"
    ]
    if feature in df.columns
]


if len(target_candidates) == 0:

    raise KeyError(
        "No target feature was found. "
        "Expected is_fraud or is_fraude."
    )


if len(target_candidates) > 1:

    raise ValueError(
        "Multiple target feature candidates were found: "
        f"{target_candidates}"
    )


target_source_feature = (
    target_candidates[0]
)


# ============================================================
# 08. VALIDATE THE NID IDENTIFIER
# ============================================================

nid_numeric = pd.to_numeric(
    df["NID"],
    errors="coerce"
)


if nid_numeric.isna().any():

    raise ValueError(
        "NID contains missing or non-numeric values."
    )


expected_nid = np.arange(
    1,
    len(df) + 1,
    dtype=np.int64
)


actual_nid = (
    nid_numeric
    .to_numpy(
        dtype=np.int64
    )
)


if not np.array_equal(
    actual_nid,
    expected_nid
):

    raise ValueError(
        "NID is not a complete sequential identifier "
        "from 1 to the number of observations."
    )


if not nid_numeric.is_unique:

    raise ValueError(
        "Duplicated NID values were detected."
    )


df["NID"] = actual_nid


print(
    "\nINITIAL NID VALIDATION"
)

print(
    "=" * 100
)

print(
    "First NID:",
    int(
        df["NID"].iloc[0]
    )
)

print(
    "Last NID:",
    int(
        df["NID"].iloc[-1]
    )
)

print(
    "Unique NIDs:",
    int(
        df["NID"].nunique()
    )
)

print(
    "Duplicated NIDs:",
    int(
        df["NID"]
        .duplicated()
        .sum()
    )
)


del expected_nid
del actual_nid
del nid_numeric

gc.collect()


# ============================================================
# 09. REMOVE UNUSED FEATURES
# ============================================================

columns_to_remove = [
    "city",
    "state",
    "zip",
    "trans_num",
    "unix_time",
    "street"
]


df = df.drop(
    columns=columns_to_remove
)


print(
    "\nUNUSED FEATURES REMOVED"
)

print(
    "=" * 100
)

for column in columns_to_remove:

    print(
        column
    )


# ============================================================
# 10. RENAME AND VALIDATE THE TARGET FEATURE
# ============================================================

df = df.rename(
    columns={
        target_source_feature:
            "TARGET"
    }
)


target_numeric = pd.to_numeric(
    df["TARGET"],
    errors="coerce"
)


if target_numeric.isna().any():

    raise ValueError(
        "The target contains missing or non-numeric values."
    )


unexpected_target_values = sorted(
    set(
        target_numeric.unique()
    )
    -
    {
        0,
        1
    }
)


if unexpected_target_values:

    raise ValueError(
        "Unexpected target values detected: "
        f"{unexpected_target_values}"
    )


df["TARGET"] = (
    target_numeric
    .astype(
        np.int8
    )
)


del target_numeric


# ============================================================
# 11. PARSE AND VALIDATE TRANSACTION DATE AND TIME
# ============================================================

transaction_datetime = pd.to_datetime(
    df["trans_date_trans_time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)


invalid_transaction_datetime = (
    transaction_datetime.isna()
)


invalid_transaction_datetime_count = int(
    invalid_transaction_datetime.sum()
)


if invalid_transaction_datetime_count > 0:

    invalid_examples = (
        df.loc[
            invalid_transaction_datetime,
            "trans_date_trans_time"
        ]
        .head(
            10
        )
        .tolist()
    )


    raise ValueError(
        "Invalid or missing transaction date/time values detected. "
        f"Count: {invalid_transaction_datetime_count}. "
        f"Examples: {invalid_examples}"
    )


df["trans_date_trans_time"] = (
    transaction_datetime
)


# ============================================================
# 12. CREATE TRANSACTION DAY
# ============================================================

df["TRANS_DAY"] = (
    df["trans_date_trans_time"]
    .dt.day
)


# ============================================================
# 13. CREATE TRANSACTION WEEKDAY
# ============================================================

weekday_mapping = {
    0: "Monday",
    1: "Tuesday",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday"
}


df["TRANS_WEEK"] = (
    df["trans_date_trans_time"]
    .dt.dayofweek
    .map(
        weekday_mapping
    )
)


# ============================================================
# 14. CREATE TRANSACTION YEAR
# ============================================================

df["TRANS_YEAR"] = (
    df["trans_date_trans_time"]
    .dt.year
)


unexpected_years = sorted(
    set(
        df["TRANS_YEAR"]
        .unique()
    )
    -
    {
        2019,
        2020
    }
)


if unexpected_years:

    raise ValueError(
        "Unexpected transaction years detected: "
        f"{unexpected_years}"
    )


# ============================================================
# 15. CREATE CYCLICAL MONTH REPRESENTATION
# ============================================================

month = (
    df["trans_date_trans_time"]
    .dt.month
)


df["TRANS_MONTH_SIN"] = np.sin(
    2
    *
    np.pi
    *
    (
        month
        -
        1
    )
    /
    12
)


df["TRANS_MONTH_COS"] = np.cos(
    2
    *
    np.pi
    *
    (
        month
        -
        1
    )
    /
    12
)


# ============================================================
# 16. CREATE CYCLICAL TIME REPRESENTATION
# ============================================================

decimal_hour = (

    df[
        "trans_date_trans_time"
    ]
    .dt.hour

    +

    df[
        "trans_date_trans_time"
    ]
    .dt.minute
    /
    60

    +

    df[
        "trans_date_trans_time"
    ]
    .dt.second
    /
    3600
)


df["TRANS_HOUR_SIN"] = np.sin(
    2
    *
    np.pi
    *
    decimal_hour
    /
    24
)


df["TRANS_HOUR_COS"] = np.cos(
    2
    *
    np.pi
    *
    decimal_hour
    /
    24
)


# ============================================================
# 17. VALIDATE TRANSACTION-DERIVED FEATURES
# ============================================================

if not df[
    "TRANS_DAY"
].between(
    1,
    31
).all():

    raise ValueError(
        "TRANS_DAY contains values outside the interval 1 to 31."
    )


expected_weekdays = set(
    weekday_mapping.values()
)


actual_weekdays = set(
    df[
        "TRANS_WEEK"
    ]
    .unique()
)


unexpected_weekdays = (
    actual_weekdays
    -
    expected_weekdays
)


if unexpected_weekdays:

    raise ValueError(
        "Unexpected weekday values detected: "
        f"{sorted(unexpected_weekdays)}"
    )


cyclical_features = [
    "TRANS_MONTH_SIN",
    "TRANS_MONTH_COS",
    "TRANS_HOUR_SIN",
    "TRANS_HOUR_COS"
]


for feature in cyclical_features:

    if not df[
        feature
    ].between(
        -1.0,
        1.0
    ).all():

        raise ValueError(
            f"{feature} contains values outside [-1, 1]."
        )


month_cyclical_norm = (
    df["TRANS_MONTH_SIN"] ** 2
    +
    df["TRANS_MONTH_COS"] ** 2
)


hour_cyclical_norm = (
    df["TRANS_HOUR_SIN"] ** 2
    +
    df["TRANS_HOUR_COS"] ** 2
)


if not np.allclose(
    month_cyclical_norm,
    1.0,
    atol=1e-10
):

    raise ValueError(
        "Month sine/cosine encoding failed the unit-circle check."
    )


if not np.allclose(
    hour_cyclical_norm,
    1.0,
    atol=1e-10
):

    raise ValueError(
        "Hour sine/cosine encoding failed the unit-circle check."
    )


del month
del decimal_hour
del month_cyclical_norm
del hour_cyclical_norm
del transaction_datetime
del invalid_transaction_datetime

gc.collect()


# ============================================================
# 18. REMOVE THE ORIGINAL TRANSACTION DATETIME FEATURE
# ============================================================

df = df.drop(
    columns=[
        "trans_date_trans_time"
    ]
)


# ============================================================
# 19. RENAME TRANSACTION AND SENDER FEATURES
# ============================================================

df = df.rename(
    columns={
        "amt":
            "TRANS_VALUE",

        "cc_num":
            "TRANS_NUM_CARD",

        "job":
            "SEND_JOB",

        "gender":
            "SEND_GENDER",

        "lat":
            "SEND_LAT_REGISTER",

        "long":
            "SEND_LONG_REGISTER",

        "city_pop":
            "SEND_POP_REGISTER"
    }
)


# ============================================================
# 20. VALIDATE SENDER GENDER
# ============================================================

unexpected_gender_values = sorted(
    set(
        df[
            "SEND_GENDER"
        ]
        .dropna()
        .astype(
            str
        )
        .unique()
    )
    -
    {
        "F",
        "M"
    }
)


if unexpected_gender_values:

    raise ValueError(
        "Unexpected SEND_GENDER values detected: "
        f"{unexpected_gender_values}"
    )


if df[
    "SEND_GENDER"
].isna().any():

    raise ValueError(
        "Missing SEND_GENDER values were detected."
    )


# ============================================================
# 21. CREATE THE SENDER FULL NAME
# ============================================================

missing_name_values = int(
    (
        df["first"].isna()
        |
        df["last"].isna()
    )
    .sum()
)


if missing_name_values > 0:

    raise ValueError(
        "Missing sender first or last names were detected. "
        f"Count: {missing_name_values}"
    )


df["SEND_NAME"] = (
    df["first"]
    .astype(
        str
    )
    .str.strip()

    +

    " "

    +

    df["last"]
    .astype(
        str
    )
    .str.strip()
)


df["SEND_NAME"] = (
    df["SEND_NAME"]
    .str.strip()
)


if (
    df["SEND_NAME"]
    .eq("")
    .any()
):

    raise ValueError(
        "Empty SEND_NAME values were created."
    )


# ============================================================
# 22. PARSE AND VALIDATE DATE OF BIRTH
# ============================================================

birth_date = pd.to_datetime(
    df["dob"],
    errors="coerce"
)


invalid_birth_date = (
    birth_date.isna()
)


invalid_birth_date_count = int(
    invalid_birth_date.sum()
)


if invalid_birth_date_count > 0:

    invalid_birth_examples = (
        df.loc[
            invalid_birth_date,
            "dob"
        ]
        .head(
            10
        )
        .tolist()
    )


    raise ValueError(
        "Invalid or missing date-of-birth values detected. "
        f"Count: {invalid_birth_date_count}. "
        f"Examples: {invalid_birth_examples}"
    )


df["dob"] = (
    birth_date
)


# ============================================================
# 23. CREATE SENDER AGE USING THE FIXED REFERENCE DATE
# ============================================================

df["SEND_AGE"] = (

    (
        reference_date
        -
        df["dob"]
    )
    .dt.total_seconds()

    /

    (
        365.2425
        *
        24
        *
        60
        *
        60
    )

).round(
    2
)


# ============================================================
# 24. VALIDATE SENDER AGE
# ============================================================

if (
    df["SEND_AGE"]
    .isna()
    .any()
):

    raise ValueError(
        "Missing SEND_AGE values were created."
    )


if (
    df["SEND_AGE"]
    <
    0
).any():

    raise ValueError(
        "Negative SEND_AGE values were detected."
    )


if (
    df["SEND_AGE"]
    >
    120
).any():

    raise ValueError(
        "SEND_AGE values greater than 120 years were detected."
    )


print(
    "\nSEND_AGE VALIDATION"
)

print(
    "=" * 100
)

print(
    "Reference date:",
    reference_date.strftime(
        "%Y-%m-%d"
    )
)

print(
    "Minimum age:",
    df[
        "SEND_AGE"
    ].min()
)

print(
    "Maximum age:",
    df[
        "SEND_AGE"
    ].max()
)


del birth_date
del invalid_birth_date

gc.collect()


# ============================================================
# 25. REMOVE ORIGINAL NAME AND DATE-OF-BIRTH FEATURES
# ============================================================

df = df.drop(
    columns=[
        "first",
        "last",
        "dob"
    ]
)


# ============================================================
# 26. RENAME RECEIVER FEATURES
# ============================================================

df = df.rename(
    columns={
        "merchant":
            "RECEIVE_LOC",

        "category":
            "RECEIVE_CATEGORY",

        "merch_lat":
            "RECEIVE_LAT",

        "merch_long":
            "RECEIVE_LONG"
    }
)


# ============================================================
# 27. VALIDATE NUMERICAL DOMAIN CONSTRAINTS
# ============================================================

if (
    df["TRANS_VALUE"]
    <
    0
).any():

    raise ValueError(
        "Negative TRANS_VALUE values were detected."
    )


if (
    df["SEND_POP_REGISTER"]
    <
    0
).any():

    raise ValueError(
        "Negative SEND_POP_REGISTER values were detected."
    )


latitude_features = [
    "SEND_LAT_REGISTER",
    "RECEIVE_LAT"
]


longitude_features = [
    "SEND_LONG_REGISTER",
    "RECEIVE_LONG"
]


for feature in latitude_features:

    if not df[
        feature
    ].between(
        -90,
        90
    ).all():

        raise ValueError(
            f"{feature} contains invalid latitude values."
        )


for feature in longitude_features:

    if not df[
        feature
    ].between(
        -180,
        180
    ).all():

        raise ValueError(
            f"{feature} contains invalid longitude values."
        )


# ============================================================
# 28. ADD FUTURE ENCODER NOMENCLATURE
#
# The values are NOT encoded in this stage.
# The suffixes indicate the encoding strategy that will be
# applied later in the modeling pipeline.
# ============================================================

df = df.rename(
    columns={

        # ----------------------------------------------------
        # IDENTIFIER
        # ----------------------------------------------------

        "NID":
            "NID_ALPHA",

        # ----------------------------------------------------
        # FUTURE FREQUENCY ENCODING WITH FALLBACK
        # ----------------------------------------------------

        "TRANS_NUM_CARD":
            "TRANS_NUM_CARD_FEWF",

        "RECEIVE_LOC":
            "RECEIVE_LOC_FEWF",

        "SEND_JOB":
            "SEND_JOB_FEWF",

        "SEND_NAME":
            "SEND_NAME_FEWF",

        # ----------------------------------------------------
        # FUTURE BINARY ENCODING
        # ----------------------------------------------------

        "SEND_GENDER":
            "SEND_GENDER_BE",

        "TRANS_YEAR":
            "TRANS_YEAR_BE",

        # ----------------------------------------------------
        # FUTURE ONE-HOT ENCODING WITH IGNORE
        # ----------------------------------------------------

        "RECEIVE_CATEGORY":
            "RECEIVE_CATEGORY_OHEWI",

        "TRANS_WEEK":
            "TRANS_WEEK_OHEWI",

        # ----------------------------------------------------
        # TARGET
        # ----------------------------------------------------

        "TARGET":
            "TARGET_OMEGA"
    }
)


# ============================================================
# 29. DEFINE THE EXPECTED MEDIUM DATASET STRUCTURE
# ============================================================

expected_medium_columns = [
    "NID_ALPHA",
    "TRANS_NUM_CARD_FEWF",
    "TRANS_VALUE",
    "TRANS_DAY",
    "TRANS_WEEK_OHEWI",
    "TRANS_YEAR_BE",
    "TRANS_MONTH_SIN",
    "TRANS_MONTH_COS",
    "TRANS_HOUR_SIN",
    "TRANS_HOUR_COS",
    "SEND_NAME_FEWF",
    "SEND_GENDER_BE",
    "SEND_AGE",
    "SEND_JOB_FEWF",
    "SEND_LAT_REGISTER",
    "SEND_LONG_REGISTER",
    "SEND_POP_REGISTER",
    "RECEIVE_LOC_FEWF",
    "RECEIVE_CATEGORY_OHEWI",
    "RECEIVE_LAT",
    "RECEIVE_LONG",
    "TARGET_OMEGA"
]


# ============================================================
# 30. VALIDATE THE EXPECTED MEDIUM DATASET FEATURES
# ============================================================

missing_medium_features = [
    feature
    for feature in expected_medium_columns
    if feature not in df.columns
]


unexpected_medium_features = [
    feature
    for feature in df.columns
    if feature not in expected_medium_columns
]


if missing_medium_features:

    raise KeyError(
        "Missing medium dataset features: "
        + ", ".join(
            missing_medium_features
        )
    )


if unexpected_medium_features:

    raise KeyError(
        "Unexpected medium dataset features: "
        + ", ".join(
            unexpected_medium_features
        )
    )


# ============================================================
# 31. APPLY THE DEFINITIVE FEATURE ORDER
# ============================================================

df = df[
    expected_medium_columns
]


# ============================================================
# 32. VALIDATE FINAL ROW COUNT
# ============================================================

if len(
    df
) != initial_rows:

    raise ValueError(
        "The number of observations changed during "
        "medium dataset transformation."
    )


# ============================================================
# 33. VALIDATE FINAL IDENTIFIER
# ============================================================

if not df[
    "NID_ALPHA"
].is_unique:

    raise ValueError(
        "NID_ALPHA is not unique."
    )


if (
    df["NID_ALPHA"]
    .duplicated()
    .sum()
    !=
    0
):

    raise ValueError(
        "Duplicated NID_ALPHA values were detected."
    )


# ============================================================
# 34. VALIDATE FINAL MISSING VALUES
# ============================================================

missing_values = (
    df.isna()
    .sum()
)


features_with_missing_values = (
    missing_values[
        missing_values
        >
        0
    ]
)


if not features_with_missing_values.empty:

    raise ValueError(
        "Missing values were detected in the medium dataset:\n"
        f"{features_with_missing_values}"
    )


# ============================================================
# 35. VALIDATE FINAL NUMERICAL VALUES
# ============================================================

numerical_features = (
    df.select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
)


for feature in numerical_features:

    feature_values = (
        df[
            feature
        ]
        .to_numpy()
    )


    if not np.isfinite(
        feature_values
    ).all():

        raise ValueError(
            f"{feature} contains infinite or non-finite values."
        )


# ============================================================
# 36. DISPLAY MEDIUM DATASET DIMENSIONS
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "MEDIUM DATASET"
)

print(
    "=" * 100
)

print(
    "Rows:",
    df.shape[0]
)

print(
    "Features:",
    df.shape[1]
)


# ============================================================
# 37. DISPLAY FINAL FEATURES
# ============================================================

print(
    "\nFINAL FEATURES"
)

print(
    "=" * 100
)


for index, column in enumerate(
    df.columns,
    start=1
):

    print(
        f"{index}. {column}"
    )


# ============================================================
# 38. DISPLAY TARGET DISTRIBUTION
# ============================================================

print(
    "\nTARGET DISTRIBUTION"
)

print(
    "=" * 100
)


print(
    df[
        "TARGET_OMEGA"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)


print(
    "\nTARGET PROPORTIONS"
)

print(
    "=" * 100
)


print(
    df[
        "TARGET_OMEGA"
    ]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .sort_index()
)


# ============================================================
# 39. REMOVE ANY PREVIOUS TEMPORARY OUTPUT
# ============================================================

if temporary_medium_dataset_file.exists():

    temporary_medium_dataset_file.unlink()

    print(
        "\nPrevious temporary medium dataset removed."
    )


# ============================================================
# 40. SAVE THE REFRESHED MEDIUM DATASET TEMPORARILY
# ============================================================

print(
    "\nSaving refreshed medium dataset..."
)


df.to_csv(
    temporary_medium_dataset_file,
    index=False
)


# ============================================================
# 41. VALIDATE THE TEMPORARY OUTPUT
# ============================================================

temporary_file_is_valid = (
    temporary_medium_dataset_file.exists()
    and
    temporary_medium_dataset_file.is_file()
    and
    temporary_medium_dataset_file.stat().st_size > 0
)


if not temporary_file_is_valid:

    raise FileNotFoundError(
        "The temporary medium dataset was not "
        "created correctly."
    )


temporary_header = (
    pd.read_csv(
        temporary_medium_dataset_file,
        nrows=0
    )
    .columns
    .tolist()
)


if temporary_header != expected_medium_columns:

    raise ValueError(
        "The saved temporary medium dataset has "
        "an unexpected column structure."
    )


print(
    "Temporary medium dataset validated successfully."
)


# ============================================================
# 42. REPLACE THE PREVIOUS MEDIUM DATASET
# ============================================================

medium_dataset_already_existed = (
    medium_dataset_file.exists()
)


temporary_medium_dataset_file.replace(
    medium_dataset_file
)


# ============================================================
# 43. VALIDATE THE FINAL OUTPUT FILE
# ============================================================

medium_file_is_valid = (
    medium_dataset_file.exists()
    and
    medium_dataset_file.is_file()
    and
    medium_dataset_file.stat().st_size > 0
)


if not medium_file_is_valid:

    raise FileNotFoundError(
        "dataset_medium.csv was not created correctly."
    )


# ============================================================
# 44. DISPLAY FINAL OUTPUT INFORMATION
# ============================================================

print(
    "\nMEDIUM DATASET FILE"
)

print(
    "=" * 100
)


if medium_dataset_already_existed:

    print(
        "Previous dataset_medium.csv overwritten successfully."
    )

else:

    print(
        "dataset_medium.csv created successfully."
    )


print(
    "Location:",
    medium_dataset_file
)


print(
    "File size:",
    f"{medium_dataset_file.stat().st_size / (1024 ** 2):.2f} MB"
)


# ============================================================
# 45. FINAL CONFIRMATION
# ============================================================

print(
    "\nMEDIUM DATASET CREATION COMPLETED"
)

print(
    "=" * 100
)

print(
    "Observations:",
    df.shape[0]
)

print(
    "Features:",
    df.shape[1]
)

print(
    "Output:",
    medium_dataset_file
)


# ============================================================
# 46. RELEASE MEMORY
# ============================================================

del df

gc.collect()


print(
    "\nMedium dataset DataFrame released from memory."
)


Primary dataset file validated successfully.

Loading the primary dataset...

PRIMARY DATASET DIMENSIONS
Rows: 1852394
Features: 23

INITIAL NID VALIDATION
First NID: 1
Last NID: 1852394
Unique NIDs: 1852394
Duplicated NIDs: 0

UNUSED FEATURES REMOVED
city
state
zip
trans_num
unix_time
street

SEND_AGE VALIDATION
Reference date: 2026-09-03
Minimum age: 21.59
Maximum age: 101.84

MEDIUM DATASET
Rows: 1852394
Features: 22

FINAL FEATURES
1. NID_ALPHA
2. TRANS_NUM_CARD_FEWF
3. TRANS_VALUE
4. TRANS_DAY
5. TRANS_WEEK_OHEWI
6. TRANS_YEAR_BE
7. TRANS_MONTH_SIN
8. TRANS_MONTH_COS
9. TRANS_HOUR_SIN
10. TRANS_HOUR_COS
11. SEND_NAME_FEWF
12. SEND_GENDER_BE
13. SEND_AGE
14. SEND_JOB_FEWF
15. SEND_LAT_REGISTER
16. SEND_LONG_REGISTER
17. SEND_POP_REGISTER
18. RECEIVE_LOC_FEWF
19. RECEIVE_CATEGORY_OHEWI
20. RECEIVE_LAT
21. RECEIVE_LONG
22. TARGET_OMEGA

TARGET DISTRIBUTION
TARGET_OMEGA
0    1842743
1       9651
Name: count, dtype: int64

TARGET PROPORTIONS
TARGET_OMEGA
0    0.99479
1    0.00521
Name

## <span style="color:blue">FINAL DATASET MEMORY OPTIMIZATION AND PARQUET CREATION</span>

In [5]:
# ============================================================
# 01. SETTINGS
# ============================================================

project_root = Path(
    "/projeto_tcc_2026"
)

input_file = (
    project_root
    / "data"
    / "medium_dataset"
    / "dataset_medium.csv"
)

output_folder = (
    project_root
    / "data"
    / "final_dataset"
)

output_file = (
    output_folder
    / "dataset_final.parquet"
)

temporary_output_file = (
    output_folder
    / "_dataset_final.parquet"
)

backup_output_file = (
    output_folder
    / "_dataset_final_previous.parquet"
)

category_unique_ratio_threshold = 0.05

float_relative_tolerance = 1e-6

float_absolute_tolerance = 1e-7


# ============================================================
# 02. ENSURE THE OUTPUT DIRECTORY EXISTS
# ============================================================

output_folder.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 03. RECOVER OR REMOVE STALE BACKUP FILE
# ============================================================

if backup_output_file.exists():

    if output_file.exists():

        backup_output_file.unlink()

        print(
            "\nStale backup file removed."
        )

    else:

        shutil.move(
            str(
                backup_output_file
            ),
            str(
                output_file
            )
        )

        print(
            "\nPrevious final dataset restored from backup."
        )


# ============================================================
# 04. REMOVE STALE TEMPORARY OUTPUT
# ============================================================

if temporary_output_file.exists():

    temporary_output_file.unlink()

    print(
        "\nPrevious temporary Parquet file removed."
    )


# ============================================================
# 05. VALIDATE THE MEDIUM DATASET FILE
# ============================================================

input_file_is_valid = (
    input_file.exists()
    and
    input_file.is_file()
    and
    input_file.stat().st_size > 0
)


if not input_file_is_valid:

    raise FileNotFoundError(
        "dataset_medium.csv is missing or empty: "
        f"{input_file}"
    )


print(
    "\nMedium dataset file validated successfully."
)


# ============================================================
# 06. LOAD THE MEDIUM DATASET
# ============================================================

print(
    "\nLoading the medium dataset..."
)


df = pd.read_csv(
    input_file,
    low_memory=False
)


if df.empty:

    raise ValueError(
        "The medium dataset is empty."
    )


print(
    "Dataset loaded successfully."
)


# ============================================================
# 07. DEFINE THE EXPECTED FINAL DATASET STRUCTURE
# ============================================================

expected_final_columns = [
    "NID_ALPHA",
    "TRANS_NUM_CARD_FEWF",
    "TRANS_VALUE",
    "TRANS_DAY",
    "TRANS_WEEK_OHEWI",
    "TRANS_YEAR_BE",
    "TRANS_MONTH_SIN",
    "TRANS_MONTH_COS",
    "TRANS_HOUR_SIN",
    "TRANS_HOUR_COS",
    "SEND_NAME_FEWF",
    "SEND_GENDER_BE",
    "SEND_AGE",
    "SEND_JOB_FEWF",
    "SEND_LAT_REGISTER",
    "SEND_LONG_REGISTER",
    "SEND_POP_REGISTER",
    "RECEIVE_LOC_FEWF",
    "RECEIVE_CATEGORY_OHEWI",
    "RECEIVE_LAT",
    "RECEIVE_LONG",
    "TARGET_OMEGA"
]


# ============================================================
# 08. CHECK FOR DUPLICATED COLUMN NAMES
# ============================================================

duplicated_columns = (
    df.columns[
        df.columns.duplicated()
    ]
    .tolist()
)


if duplicated_columns:

    raise ValueError(
        "Duplicated column names detected: "
        f"{duplicated_columns}"
    )


# ============================================================
# 09. VALIDATE THE DATASET FEATURES
# ============================================================

actual_columns = list(
    df.columns
)


missing_columns = [
    column
    for column in expected_final_columns
    if column not in actual_columns
]


unexpected_columns = [
    column
    for column in actual_columns
    if column not in expected_final_columns
]


if missing_columns:

    raise KeyError(
        "Missing required features: "
        + ", ".join(
            missing_columns
        )
    )


if unexpected_columns:

    raise KeyError(
        "Unexpected features detected: "
        + ", ".join(
            unexpected_columns
        )
    )


if actual_columns != expected_final_columns:

    raise ValueError(
        "The medium dataset contains the expected features "
        "but not in the expected order."
    )


print(
    "\nFinal feature structure validated successfully."
)


# ============================================================
# 10. RECORD DATASET DIMENSIONS
# ============================================================

rows_before = int(
    df.shape[0]
)

features_before = int(
    df.shape[1]
)


print(
    "\nMEDIUM DATASET DIMENSIONS"
)

print(
    "=" * 100
)

print(
    "Rows:",
    rows_before
)

print(
    "Features:",
    features_before
)


# ============================================================
# 11. VALIDATE MISSING VALUES BEFORE OPTIMIZATION
# ============================================================

missing_values_before = (
    df.isna()
    .sum()
)


features_with_missing_values = (
    missing_values_before[
        missing_values_before
        >
        0
    ]
)


if not features_with_missing_values.empty:

    raise ValueError(
        "Missing values were detected before optimization:\n"
        f"{features_with_missing_values}"
    )


print(
    "\nNo missing values detected."
)


# ============================================================
# 12. VALIDATE THE UNIQUE IDENTIFIER
# ============================================================

nid_numeric = pd.to_numeric(
    df[
        "NID_ALPHA"
    ],
    errors="coerce"
)


if nid_numeric.isna().any():

    raise ValueError(
        "NID_ALPHA contains missing or non-numeric values."
    )


nid_values_float = (
    nid_numeric
    .to_numpy(
        dtype=np.float64
    )
)


if not np.equal(
    nid_values_float,
    np.floor(
        nid_values_float
    )
).all():

    raise ValueError(
        "NID_ALPHA contains non-integer values."
    )


actual_nid = (
    nid_numeric
    .to_numpy(
        dtype=np.int64
    )
)


expected_nid = np.arange(
    1,
    rows_before + 1,
    dtype=np.int64
)


if not np.array_equal(
    actual_nid,
    expected_nid
):

    raise ValueError(
        "NID_ALPHA is not a complete sequential identifier "
        "from 1 to the number of observations."
    )


if len(
    np.unique(
        actual_nid
    )
) != rows_before:

    raise ValueError(
        "Duplicated NID_ALPHA values were detected."
    )


print(
    "\nNID_ALPHA VALIDATION"
)

print(
    "=" * 100
)

print(
    "First NID_ALPHA:",
    int(
        actual_nid[0]
    )
)

print(
    "Last NID_ALPHA:",
    int(
        actual_nid[-1]
    )
)

print(
    "Unique identifiers:",
    rows_before
)

print(
    "Duplicated identifiers:",
    0
)


del nid_numeric
del nid_values_float
del actual_nid
del expected_nid

gc.collect()


# ============================================================
# 13. VALIDATE THE TARGET
# ============================================================

target_numeric = pd.to_numeric(
    df[
        "TARGET_OMEGA"
    ],
    errors="coerce"
)


if target_numeric.isna().any():

    raise ValueError(
        "TARGET_OMEGA contains missing or non-numeric values."
    )


target_unique_values = set(
    target_numeric.unique()
)


unexpected_target_values = sorted(
    target_unique_values
    -
    {
        0,
        1
    }
)


if unexpected_target_values:

    raise ValueError(
        "Unexpected TARGET_OMEGA values detected: "
        f"{unexpected_target_values}"
    )


print(
    "\nTARGET VALIDATION"
)

print(
    "=" * 100
)

print(
    target_numeric
    .value_counts()
    .sort_index()
)


del target_numeric
del target_unique_values

gc.collect()


# ============================================================
# 14. VALIDATE NUMERICAL VALUES BEFORE OPTIMIZATION
# ============================================================

numerical_columns_before = (
    df.select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
    .tolist()
)


for column in numerical_columns_before:

    values = (
        df[
            column
        ]
        .to_numpy()
    )


    if not np.isfinite(
        values
    ).all():

        raise ValueError(
            f"{column} contains non-finite numerical values."
        )


print(
    "\nAll numerical features contain finite values."
)


# ============================================================
# 15. RECORD INFORMATION BEFORE OPTIMIZATION
# ============================================================

types_before = (
    df.dtypes
    .astype(str)
    .copy()
)


unique_values_before = (
    df.nunique(
        dropna=False
    )
)


column_memory_before = (
    df.memory_usage(
        index=False,
        deep=True
    )
    .copy()
)


total_memory_before = int(
    column_memory_before.sum()
)


disk_size_before = int(
    input_file.stat().st_size
)


# ============================================================
# 16. OPTIMIZE FLOATING-POINT FEATURES
#
# float64 -> float32 only when:
#
# 1. Values remain within the defined numerical tolerance.
# 2. Feature cardinality is preserved.
# ============================================================

print(
    "\nOptimizing floating-point features..."
)


float_columns = (
    df.select_dtypes(
        include=[
            "floating"
        ]
    )
    .columns
    .tolist()
)


float_downcast_records = []


for column in float_columns:

    original_series = (
        df[
            column
        ]
    )


    original_dtype = str(
        original_series.dtype
    )


    original_values = (
        original_series
        .to_numpy(
            dtype=np.float64,
            copy=True
        )
    )


    candidate_series = pd.to_numeric(
        original_series,
        downcast="float"
    )


    candidate_dtype = str(
        candidate_series.dtype
    )


    candidate_values = (
        candidate_series
        .to_numpy(
            dtype=np.float64
        )
    )


    values_are_close = bool(
        np.allclose(
            original_values,
            candidate_values,
            rtol=float_relative_tolerance,
            atol=float_absolute_tolerance,
            equal_nan=True
        )
    )


    original_unique_count = int(
        original_series.nunique(
            dropna=False
        )
    )


    candidate_unique_count = int(
        candidate_series.nunique(
            dropna=False
        )
    )


    cardinality_preserved = (
        original_unique_count
        ==
        candidate_unique_count
    )


    absolute_differences = np.abs(
        original_values
        -
        candidate_values
    )


    max_absolute_difference = float(
        np.max(
            absolute_differences
        )
    )


    downcast_applied = (
        values_are_close
        and
        cardinality_preserved
        and
        candidate_dtype
        !=
        original_dtype
    )


    if downcast_applied:

        df[
            column
        ] = (
            candidate_series
        )


    float_downcast_records.append({

        "FEATURE":
            column,

        "TYPE_BEFORE":
            original_dtype,

        "CANDIDATE_TYPE":
            candidate_dtype,

        "MAX_ABSOLUTE_DIFFERENCE":
            max_absolute_difference,

        "CARDINALITY_BEFORE":
            original_unique_count,

        "CARDINALITY_AFTER":
            candidate_unique_count,

        "CARDINALITY_PRESERVED":
            cardinality_preserved,

        "WITHIN_NUMERICAL_TOLERANCE":
            values_are_close,

        "DOWNCAST_APPLIED":
            downcast_applied
    })


    del original_values
    del candidate_values
    del absolute_differences
    del candidate_series


float_downcast_table = pd.DataFrame(
    float_downcast_records
)


gc.collect()


# ============================================================
# 17. OPTIMIZE INTEGER FEATURES
#
# Integer downcasting is exact.
# Pandas selects the smallest integer dtype capable of
# preserving the complete numerical range.
# ============================================================

print(
    "Optimizing integer features..."
)


integer_columns = (
    df.select_dtypes(
        include=[
            "integer"
        ]
    )
    .columns
    .tolist()
)


integer_downcast_records = []


for column in integer_columns:

    original_series = (
        df[
            column
        ]
    )


    original_dtype = str(
        original_series.dtype
    )


    candidate_series = pd.to_numeric(
        original_series,
        downcast="integer"
    )


    candidate_dtype = str(
        candidate_series.dtype
    )


    values_preserved = bool(
        np.array_equal(
            original_series.to_numpy(),
            candidate_series.to_numpy()
        )
    )


    if not values_preserved:

        raise ValueError(
            f"Integer optimization changed values in {column}."
        )


    df[
        column
    ] = (
        candidate_series
    )


    integer_downcast_records.append({

        "FEATURE":
            column,

        "TYPE_BEFORE":
            original_dtype,

        "TYPE_AFTER":
            candidate_dtype,

        "VALUES_PRESERVED":
            values_preserved,

        "TYPE_CHANGED":
            original_dtype
            !=
            candidate_dtype
    })


integer_downcast_table = pd.DataFrame(
    integer_downcast_records
)


gc.collect()


# ============================================================
# 18. OPTIMIZE TEXT FEATURES WITH CATEGORY DTYPE
#
# Conversion is applied only when:
#
# 1. The proportion of unique values is <= 5%.
# 2. Category representation actually consumes less memory.
#
# This is a storage optimization only.
# No FEWF, OHEWI, or binary encoding is applied here.
# ============================================================

print(
    "Analyzing categorical features..."
)


text_columns = (
    df.select_dtypes(
        include=[
            "object",
            "string"
        ]
    )
    .columns
    .tolist()
)


category_optimization_records = []

columns_converted_to_category = []


for column in text_columns:

    original_series = (
        df[
            column
        ]
    )


    unique_count = int(
        original_series.nunique(
            dropna=False
        )
    )


    unique_ratio = (
        unique_count
        /
        rows_before
    )


    original_memory = int(
        original_series.memory_usage(
            index=False,
            deep=True
        )
    )


    category_candidate = (
        original_series
        .astype(
            "category"
        )
    )


    category_memory = int(
        category_candidate.memory_usage(
            index=False,
            deep=True
        )
    )


    memory_saved = (
        original_memory
        -
        category_memory
    )


    conversion_applied = (
        unique_ratio
        <=
        category_unique_ratio_threshold
        and
        category_memory
        <
        original_memory
    )


    if conversion_applied:

        df[
            column
        ] = (
            category_candidate
        )


        columns_converted_to_category.append(
            column
        )


    category_optimization_records.append({

        "FEATURE":
            column,

        "UNIQUE_VALUES":
            unique_count,

        "UNIQUE_RATIO":
            unique_ratio,

        "MEMORY_BEFORE_MB":
            original_memory
            /
            1024**2,

        "CATEGORY_MEMORY_MB":
            category_memory
            /
            1024**2,

        "MEMORY_SAVED_MB":
            memory_saved
            /
            1024**2,

        "CONVERSION_APPLIED":
            conversion_applied
    })


    del category_candidate


category_optimization_table = pd.DataFrame(
    category_optimization_records
)


gc.collect()


# ============================================================
# 19. VALIDATE DATASET DIMENSIONS AFTER OPTIMIZATION
# ============================================================

if df.shape[0] != rows_before:

    raise ValueError(
        "The number of observations changed during "
        "type optimization."
    )


if df.shape[1] != features_before:

    raise ValueError(
        "The number of features changed during "
        "type optimization."
    )


if list(
    df.columns
) != expected_final_columns:

    raise ValueError(
        "Feature structure changed during type optimization."
    )


# ============================================================
# 20. VALIDATE MISSING VALUES AFTER OPTIMIZATION
# ============================================================

missing_values_after = (
    df.isna()
    .sum()
)


if not missing_values_after.equals(
    missing_values_before
):

    raise ValueError(
        "Missing-value structure changed during optimization."
    )


# ============================================================
# 21. VALIDATE FEATURE CARDINALITY AFTER OPTIMIZATION
# ============================================================

unique_values_after = (
    df.nunique(
        dropna=False
    )
)


cardinality_changes = (
    unique_values_after
    !=
    unique_values_before
)


if cardinality_changes.any():

    changed_features = pd.DataFrame({

        "UNIQUE_BEFORE":
            unique_values_before[
                cardinality_changes
            ],

        "UNIQUE_AFTER":
            unique_values_after[
                cardinality_changes
            ]
    })


    raise ValueError(
        "Feature cardinality changed during optimization:\n"
        f"{changed_features}"
    )


# ============================================================
# 22. VALIDATE NUMERICAL VALUES AFTER OPTIMIZATION
# ============================================================

numerical_columns_after = (
    df.select_dtypes(
        include=[
            np.number
        ]
    )
    .columns
    .tolist()
)


for column in numerical_columns_after:

    values = (
        df[
            column
        ]
        .to_numpy()
    )


    if not np.isfinite(
        values
    ).all():

        raise ValueError(
            f"{column} contains non-finite numerical values "
            "after optimization."
        )


# ============================================================
# 23. REVALIDATE THE IDENTIFIER
# ============================================================

optimized_nid = pd.to_numeric(
    df[
        "NID_ALPHA"
    ],
    errors="coerce"
)


if optimized_nid.isna().any():

    raise ValueError(
        "NID_ALPHA became invalid during optimization."
    )


optimized_nid_values = (
    optimized_nid
    .to_numpy(
        dtype=np.int64
    )
)


expected_nid = np.arange(
    1,
    rows_before + 1,
    dtype=np.int64
)


if not np.array_equal(
    optimized_nid_values,
    expected_nid
):

    raise ValueError(
        "NID_ALPHA changed during optimization."
    )


del optimized_nid
del optimized_nid_values
del expected_nid

gc.collect()


# ============================================================
# 24. REVALIDATE THE TARGET
# ============================================================

optimized_target = pd.to_numeric(
    df[
        "TARGET_OMEGA"
    ],
    errors="coerce"
)


if optimized_target.isna().any():

    raise ValueError(
        "TARGET_OMEGA became invalid during optimization."
    )


optimized_target_values = set(
    optimized_target.unique()
)


if not optimized_target_values.issubset(
    {
        0,
        1
    }
):

    raise ValueError(
        "TARGET_OMEGA contains invalid values after optimization."
    )


del optimized_target
del optimized_target_values

gc.collect()


# ============================================================
# 25. RECORD INFORMATION AFTER OPTIMIZATION
# ============================================================

types_after = (
    df.dtypes
    .astype(str)
    .copy()
)


column_memory_after = (
    df.memory_usage(
        index=False,
        deep=True
    )
    .copy()
)


total_memory_after = int(
    column_memory_after.sum()
)


# ============================================================
# 26. CREATE THE FEATURE OPTIMIZATION SUMMARY
# ============================================================

feature_summary = pd.DataFrame({

    "TYPE_BEFORE":
        types_before,

    "TYPE_AFTER":
        types_after,

    "UNIQUE_VALUES":
        unique_values_after,

    "MEMORY_BEFORE_MB":
        column_memory_before
        /
        1024**2,

    "MEMORY_AFTER_MB":
        column_memory_after
        /
        1024**2
})


feature_summary[
    "MEMORY_SAVED_MB"
] = (

    feature_summary[
        "MEMORY_BEFORE_MB"
    ]

    -

    feature_summary[
        "MEMORY_AFTER_MB"
    ]
)


feature_summary[
    "REDUCTION_%"
] = np.where(

    feature_summary[
        "MEMORY_BEFORE_MB"
    ]
    >
    0,

    (
        feature_summary[
            "MEMORY_SAVED_MB"
        ]

        /

        feature_summary[
            "MEMORY_BEFORE_MB"
        ]

        *
        100
    ),

    0.0
)


feature_summary[
    "TYPE_CHANGED"
] = (

    feature_summary[
        "TYPE_BEFORE"
    ]

    !=

    feature_summary[
        "TYPE_AFTER"
    ]
)


float_difference_lookup = {

    record[
        "FEATURE"
    ]:
        record[
            "MAX_ABSOLUTE_DIFFERENCE"
        ]

    for record in float_downcast_records
}


feature_summary[
    "MAX_ABSOLUTE_FLOAT_DIFFERENCE"
] = [

    float_difference_lookup.get(
        feature,
        np.nan
    )

    for feature in feature_summary.index
]


# ============================================================
# 27. SAVE THE NEW DATASET TO A TEMPORARY PARQUET
# ============================================================

print(
    "\nSaving refreshed final dataset..."
)


df.to_parquet(
    temporary_output_file,
    engine="pyarrow",
    compression="zstd",
    index=False
)


# ============================================================
# 28. VALIDATE THE TEMPORARY PARQUET FILE
# ============================================================

temporary_file_is_valid = (
    temporary_output_file.exists()
    and
    temporary_output_file.is_file()
    and
    temporary_output_file.stat().st_size > 0
)


if not temporary_file_is_valid:

    raise FileNotFoundError(
        "The temporary final Parquet file was not "
        "created correctly."
    )


temporary_file_size = int(
    temporary_output_file.stat().st_size
)


temporary_parquet = pq.ParquetFile(
    temporary_output_file
)


temporary_parquet_rows = int(
    temporary_parquet.metadata.num_rows
)


temporary_parquet_columns = int(
    temporary_parquet.metadata.num_columns
)


temporary_parquet_feature_names = (
    temporary_parquet
    .schema_arrow
    .names
)


if temporary_parquet_rows != rows_before:

    raise ValueError(
        "Temporary Parquet row count does not match "
        "the source dataset."
    )


if temporary_parquet_columns != features_before:

    raise ValueError(
        "Temporary Parquet column count does not match "
        "the source dataset."
    )


if temporary_parquet_feature_names != expected_final_columns:

    raise ValueError(
        "Temporary Parquet feature structure does not match "
        "the expected final structure."
    )


print(
    "Temporary Parquet validated successfully."
)

print(
    "Temporary rows:",
    temporary_parquet_rows
)

print(
    "Temporary features:",
    temporary_parquet_columns
)

print(
    "Temporary file size:",
    f"{temporary_file_size / 1024**2:.2f} MB"
)


# ============================================================
# 29. CLOSE THE TEMPORARY PARQUET READER
# ============================================================

del temporary_parquet

gc.collect()


# ============================================================
# 30. PREPARE SAFE FINAL DATASET REPLACEMENT
# ============================================================

final_dataset_already_existed = (
    output_file.exists()
)


if backup_output_file.exists():

    backup_output_file.unlink()


if final_dataset_already_existed:

    shutil.move(
        str(
            output_file
        ),
        str(
            backup_output_file
        )
    )


# ============================================================
# 31. REPLACE THE PREVIOUS FINAL DATASET
# ============================================================

print(
    "\nReplacing the previous final dataset..."
)


try:

    shutil.move(
        str(
            temporary_output_file
        ),
        str(
            output_file
        )
    )


    # ========================================================
    # 32. VALIDATE THE NEW FINAL PARQUET FILE
    # ========================================================

    output_file_is_valid = (
        output_file.exists()
        and
        output_file.is_file()
        and
        output_file.stat().st_size > 0
    )


    if not output_file_is_valid:

        raise FileNotFoundError(
            "dataset_final.parquet was not created correctly."
        )


    final_file_size = int(
        output_file.stat().st_size
    )


    if final_file_size != temporary_file_size:

        raise ValueError(
            "The final Parquet file size differs from "
            "the validated temporary Parquet file."
        )


    final_parquet = pq.ParquetFile(
        output_file
    )


    final_parquet_rows = int(
        final_parquet.metadata.num_rows
    )


    final_parquet_columns = int(
        final_parquet.metadata.num_columns
    )


    final_parquet_feature_names = (
        final_parquet
        .schema_arrow
        .names
    )


    if final_parquet_rows != rows_before:

        raise ValueError(
            "Final Parquet row count does not match "
            "the source dataset."
        )


    if final_parquet_columns != features_before:

        raise ValueError(
            "Final Parquet column count does not match "
            "the source dataset."
        )


    if final_parquet_feature_names != expected_final_columns:

        raise ValueError(
            "Final Parquet feature structure does not match "
            "the expected final structure."
        )


    del final_parquet

    gc.collect()


except Exception:

    if output_file.exists():

        output_file.unlink()


    if backup_output_file.exists():

        shutil.move(
            str(
                backup_output_file
            ),
            str(
                output_file
            )
        )


        print(
            "\nPrevious final dataset restored after "
            "replacement failure."
        )


    raise


# ============================================================
# 33. REMOVE THE BACKUP AFTER SUCCESSFUL REPLACEMENT
# ============================================================

if backup_output_file.exists():

    backup_output_file.unlink()


print(
    "Final Parquet replacement completed successfully."
)


# ============================================================
# 34. RECORD FINAL FILE SIZE
# ============================================================

disk_size_after = int(
    output_file.stat().st_size
)


# ============================================================
# 35. CALCULATE RAM MEMORY REDUCTION
# ============================================================

memory_saved = (
    total_memory_before
    -
    total_memory_after
)


memory_reduction = (

    1

    -

    (
        total_memory_after
        /
        total_memory_before
    )

) * 100


# ============================================================
# 36. CALCULATE DISK SPACE REDUCTION
# ============================================================

disk_space_saved = (
    disk_size_before
    -
    disk_size_after
)


disk_reduction = (

    1

    -

    (
        disk_size_after
        /
        disk_size_before
    )

) * 100


# ============================================================
# 37. DISPLAY FINAL DATASET SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL OPTIMIZED DATASET SUMMARY"
)

print(
    "=" * 100
)


print(
    "\nRows:",
    df.shape[0]
)

print(
    "Features:",
    df.shape[1]
)

print(
    "Missing values:",
    int(
        df.isna()
        .sum()
        .sum()
    )
)


# ============================================================
# 38. DISPLAY RAM MEMORY REDUCTION
# ============================================================

print(
    "\n"
    + "-" * 100
)

print(
    "RAM MEMORY"
)

print(
    "-" * 100
)


print(
    "Before:",
    f"{total_memory_before / 1024**2:.2f} MB"
)

print(
    "After:",
    f"{total_memory_after / 1024**2:.2f} MB"
)

print(
    "Memory saved:",
    f"{memory_saved / 1024**2:.2f} MB"
)

print(
    "Reduction:",
    f"{memory_reduction:.2f}%"
)


# ============================================================
# 39. DISPLAY DISK SPACE REDUCTION
# ============================================================

print(
    "\n"
    + "-" * 100
)

print(
    "DISK SPACE"
)

print(
    "-" * 100
)


print(
    "Medium CSV:",
    f"{disk_size_before / 1024**2:.2f} MB"
)

print(
    "Final Parquet:",
    f"{disk_size_after / 1024**2:.2f} MB"
)

print(
    "Disk space saved:",
    f"{disk_space_saved / 1024**2:.2f} MB"
)

print(
    "Reduction:",
    f"{disk_reduction:.2f}%"
)


# ============================================================
# 40. DISPLAY DATA TYPES BEFORE OPTIMIZATION
# ============================================================

print(
    "\n"
    + "-" * 100
)

print(
    "DATA TYPES BEFORE OPTIMIZATION"
)

print(
    "-" * 100
)


display(
    types_before
    .value_counts()
    .rename(
        "COUNT"
    )
    .to_frame()
)


# ============================================================
# 41. DISPLAY DATA TYPES AFTER OPTIMIZATION
# ============================================================

print(
    "\n"
    + "-" * 100
)

print(
    "DATA TYPES AFTER OPTIMIZATION"
)

print(
    "-" * 100
)


display(
    types_after
    .value_counts()
    .rename(
        "COUNT"
    )
    .to_frame()
)


# ============================================================
# 42. DISPLAY FLOAT OPTIMIZATION DETAILS
# ============================================================

print(
    "\n"
    + "-" * 100
)

print(
    "FLOAT OPTIMIZATION DETAILS"
)

print(
    "-" * 100
)


display(
    float_downcast_table
    .round(
        10
    )
)


# ============================================================
# 43. DISPLAY INTEGER OPTIMIZATION DETAILS
# ============================================================

print(
    "\n"
    + "-" * 100
)

print(
    "INTEGER OPTIMIZATION DETAILS"
)

print(
    "-" * 100
)


display(
    integer_downcast_table
)


# ============================================================
# 44. DISPLAY CATEGORY OPTIMIZATION DETAILS
# ============================================================

print(
    "\n"
    + "-" * 100
)

print(
    "CATEGORY OPTIMIZATION DETAILS"
)

print(
    "-" * 100
)


display(
    category_optimization_table
    .round(
        6
    )
)


# ============================================================
# 45. DISPLAY FEATURES WITH CHANGED DATA TYPES
# ============================================================

print(
    "\n"
    + "-" * 100
)

print(
    "FEATURES WITH CHANGED DATA TYPES"
)

print(
    "-" * 100
)


display(

    feature_summary[
        feature_summary[
            "TYPE_CHANGED"
        ]
    ]

    .sort_values(
        "MEMORY_SAVED_MB",
        ascending=False
    )

    .round(
        10
    )
)


# ============================================================
# 46. DISPLAY COMPLETE FEATURE SUMMARY
# ============================================================

print(
    "\n"
    + "-" * 100
)

print(
    "COMPLETE FEATURE SUMMARY"
)

print(
    "-" * 100
)


display(

    feature_summary

    .sort_values(
        "MEMORY_BEFORE_MB",
        ascending=False
    )

    .round(
        10
    )
)


# ============================================================
# 47. DISPLAY FEATURES CONVERTED TO CATEGORY
# ============================================================

print(
    "\n"
    + "-" * 100
)

print(
    "FEATURES CONVERTED TO CATEGORY"
)

print(
    "-" * 100
)


if columns_converted_to_category:

    for column in columns_converted_to_category:

        print(
            f"- {column}"
        )

else:

    print(
        "No features were converted to category."
    )


# ============================================================
# 48. DISPLAY FINAL OUTPUT INFORMATION
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL DATASET"
)

print(
    "=" * 100
)


if final_dataset_already_existed:

    print(
        "Previous dataset_final.parquet overwritten successfully."
    )

else:

    print(
        "dataset_final.parquet created successfully."
    )


print(
    "Location:",
    output_file
)

print(
    "File size:",
    f"{disk_size_after / 1024**2:.2f} MB"
)

print(
    "Validated rows:",
    final_parquet_rows
)

print(
    "Validated features:",
    final_parquet_columns
)


# ============================================================
# 49. FINAL CONFIRMATION
# ============================================================

print(
    "\nFINAL DATASET CREATION COMPLETED"
)

print(
    "=" * 100
)

print(
    "Observations:",
    df.shape[0]
)

print(
    "Features:",
    df.shape[1]
)

print(
    "Output:",
    output_file
)


# ============================================================
# 50. RELEASE MEMORY
# ============================================================

del df

gc.collect()


print(
    "\nFinal dataset DataFrame released from memory."
)


Medium dataset file validated successfully.

Loading the medium dataset...
Dataset loaded successfully.

Final feature structure validated successfully.

MEDIUM DATASET DIMENSIONS
Rows: 1852394
Features: 22

No missing values detected.

NID_ALPHA VALIDATION
First NID_ALPHA: 1
Last NID_ALPHA: 1852394
Unique identifiers: 1852394
Duplicated identifiers: 0

TARGET VALIDATION
TARGET_OMEGA
0    1842743
1       9651
Name: count, dtype: int64

All numerical features contain finite values.

Optimizing floating-point features...
Optimizing integer features...
Analyzing categorical features...

Saving refreshed final dataset...
Temporary Parquet validated successfully.
Temporary rows: 1852394
Temporary features: 22
Temporary file size: 64.34 MB

Replacing the previous final dataset...
Final Parquet replacement completed successfully.

FINAL OPTIMIZED DATASET SUMMARY

Rows: 1852394
Features: 22
Missing values: 0

------------------------------------------------------------------------------------

,COUNT
float64,10
int64,6
str,6



----------------------------------------------------------------------------------------------------
DATA TYPES AFTER OPTIMIZATION
----------------------------------------------------------------------------------------------------


,COUNT
float64,7
category,6
float32,3
int32,2
int8,2
int64,1
int16,1



----------------------------------------------------------------------------------------------------
FLOAT OPTIMIZATION DETAILS
----------------------------------------------------------------------------------------------------


...


----------------------------------------------------------------------------------------------------
FEATURES WITH CHANGED DATA TYPES
----------------------------------------------------------------------------------------------------


,TYPE_BEFORE,TYPE_AFTER,UNIQUE_VALUES,MEMORY_BEFORE_MB,MEMORY_AFTER_MB,MEMORY_SAVED_MB,REDUCTION_%,TYPE_CHANGED,MAX_ABSOLUTE_FLOAT_DIFFERENCE
RECEIVE_LOC_FEWF,str,category,693,54.994631,3.553880,51.440751,93.537770,True,NaN
SEND_JOB_FEWF,str,category,497,49.874805,3.546744,46.328061,92.888705,True,NaN
SEND_NAME_FEWF,str,category,989,37.438564,3.553333,33.885231,90.508895,True,NaN
RECEIVE_CATEGORY_OHEWI,str,category,14,32.727519,1.766828,30.960691,94.601401,True,NaN
TRANS_WEEK_OHEWI,str,category,7,26.412517,1.766683,24.645834,93.311192,True,NaN
SEND_GENDER_BE,str,category,2,15.899225,1.766599,14.132627,88.888775,True,NaN
TRANS_DAY,int64,int8,31,14.132645,1.766581,12.366064,87.500000,True,NaN
TARGET_OMEGA,int64,int8,2,14.132645,1.766581,12.366064,87.500000,True,NaN
TRANS_YEAR_BE,int64,int16,2,14.132645,3.533161,10.599483,75.000000,True,NaN
NID_ALPHA,int64,int32,1852394,14.132645,7.066322,7.066322,50.000000,True,NaN



----------------------------------------------------------------------------------------------------
COMPLETE FEATURE SUMMARY
----------------------------------------------------------------------------------------------------


,TYPE_BEFORE,TYPE_AFTER,UNIQUE_VALUES,MEMORY_BEFORE_MB,MEMORY_AFTER_MB,MEMORY_SAVED_MB,REDUCTION_%,TYPE_CHANGED,MAX_ABSOLUTE_FLOAT_DIFFERENCE
RECEIVE_LOC_FEWF,str,category,693,54.994631,3.553880,51.440751,93.537770,True,NaN
SEND_JOB_FEWF,str,category,497,49.874805,3.546744,46.328061,92.888705,True,NaN
SEND_NAME_FEWF,str,category,989,37.438564,3.553333,33.885231,90.508895,True,NaN
RECEIVE_CATEGORY_OHEWI,str,category,14,32.727519,1.766828,30.960691,94.601401,True,NaN
TRANS_WEEK_OHEWI,str,category,7,26.412517,1.766683,24.645834,93.311192,True,NaN
SEND_GENDER_BE,str,category,2,15.899225,1.766599,14.132627,88.888775,True,NaN
TRANS_YEAR_BE,int64,int16,2,14.132645,3.533161,10.599483,75.000000,True,NaN
TRANS_DAY,int64,int8,31,14.132645,1.766581,12.366064,87.500000,True,NaN
TRANS_VALUE,float64,float64,60616,14.132645,14.132645,0.000000,0.000000,False,0.000000e+00
TRANS_NUM_CARD_FEWF,int64,int64,999,14.132645,14.132645,0.000000,0.000000,False,NaN



----------------------------------------------------------------------------------------------------
FEATURES CONVERTED TO CATEGORY
----------------------------------------------------------------------------------------------------
- TRANS_WEEK_OHEWI
- SEND_NAME_FEWF
- SEND_GENDER_BE
- SEND_JOB_FEWF
- RECEIVE_LOC_FEWF
- RECEIVE_CATEGORY_OHEWI

FINAL DATASET
Previous dataset_final.parquet overwritten successfully.
Location: /projeto_tcc_2026/data/final_dataset/dataset_final.parquet
File size: 64.34 MB
Validated rows: 1852394
Validated features: 22

FINAL DATASET CREATION COMPLETED
Observations: 1852394
Features: 22
Output: /projeto_tcc_2026/data/final_dataset/dataset_final.parquet

Final dataset DataFrame released from memory.


## <span style="color:blue">SUMMARY AND SAMPLING</span> ##

In [6]:
# ============================================================
# 01. SETTINGS
# ============================================================

project_root = Path(
    "/projeto_tcc_2026"
)

primary_dataset_file = (
    project_root
    / "data"
    / "initial_dataset"
    / "dataset_primary.csv"
)

final_dataset_file = (
    project_root
    / "data"
    / "final_dataset"
    / "dataset_final.parquet"
)

report_file = (
    project_root
    / "data"
    / "final_dataset"
    / "dataset_comparison_report.txt"
)

temporary_report_file = (
    project_root
    / "data"
    / "final_dataset"
    / "_dataset_comparison_report.txt"
)

backup_report_file = (
    project_root
    / "data"
    / "final_dataset"
    / "_dataset_comparison_report_previous.txt"
)

random_sample_size = 10

random_state = 42


# ============================================================
# 02. DEFINE THE EXPECTED FINAL DATASET STRUCTURE
# ============================================================

expected_final_columns = [
    "NID_ALPHA",
    "TRANS_NUM_CARD_FEWF",
    "TRANS_VALUE",
    "TRANS_DAY",
    "TRANS_WEEK_OHEWI",
    "TRANS_YEAR_BE",
    "TRANS_MONTH_SIN",
    "TRANS_MONTH_COS",
    "TRANS_HOUR_SIN",
    "TRANS_HOUR_COS",
    "SEND_NAME_FEWF",
    "SEND_GENDER_BE",
    "SEND_AGE",
    "SEND_JOB_FEWF",
    "SEND_LAT_REGISTER",
    "SEND_LONG_REGISTER",
    "SEND_POP_REGISTER",
    "RECEIVE_LOC_FEWF",
    "RECEIVE_CATEGORY_OHEWI",
    "RECEIVE_LAT",
    "RECEIVE_LONG",
    "TARGET_OMEGA"
]


# ============================================================
# 03. DEFINE FINAL FEATURE DESCRIPTIONS
# ============================================================

feature_descriptions = {

    "NID_ALPHA":
        "Unique sequential identifier assigned to each transaction.",

    "TRANS_NUM_CARD_FEWF":
        "Credit card number feature preserved in its original "
        "categorical form and prepared for future Frequency "
        "Encoding With Fallback (FEWF).",

    "TRANS_VALUE":
        "Monetary value of the transaction.",

    "TRANS_DAY":
        "Day of the month on which the transaction occurred.",

    "TRANS_WEEK_OHEWI":
        "Day-of-week feature preserved as the original weekday "
        "label and prepared for future One-Hot Encoding With "
        "Ignore (OHEWI).",

    "TRANS_YEAR_BE":
        "Transaction year preserved as the original year value "
        "and prepared for future Binary Encoding (BE).",

    "TRANS_MONTH_SIN":
        "Sine component of the cyclical representation of "
        "transaction month.",

    "TRANS_MONTH_COS":
        "Cosine component of the cyclical representation of "
        "transaction month.",

    "TRANS_HOUR_SIN":
        "Sine component of the cyclical representation of "
        "transaction time, including hour, minute, and second.",

    "TRANS_HOUR_COS":
        "Cosine component of the cyclical representation of "
        "transaction time, including hour, minute, and second.",

    "SEND_NAME_FEWF":
        "Sender full name constructed from first and last name "
        "and prepared for future Frequency Encoding With "
        "Fallback (FEWF).",

    "SEND_GENDER_BE":
        "Sender gender preserved as the original category and "
        "prepared for future Binary Encoding (BE).",

    "SEND_AGE":
        "Sender age in decimal years calculated from date of "
        "birth using the fixed reference date 2026-09-03.",

    "SEND_JOB_FEWF":
        "Sender occupation preserved in its original categorical "
        "form and prepared for future Frequency Encoding With "
        "Fallback (FEWF).",

    "SEND_LAT_REGISTER":
        "Registered latitude associated with the sender.",

    "SEND_LONG_REGISTER":
        "Registered longitude associated with the sender.",

    "SEND_POP_REGISTER":
        "Population of the sender's registered city.",

    "RECEIVE_LOC_FEWF":
        "Merchant or transaction receiver preserved in its "
        "original categorical form and prepared for future "
        "Frequency Encoding With Fallback (FEWF).",

    "RECEIVE_CATEGORY_OHEWI":
        "Transaction category preserved as the original category "
        "and prepared for future One-Hot Encoding With Ignore "
        "(OHEWI).",

    "RECEIVE_LAT":
        "Latitude associated with the transaction receiver "
        "or merchant.",

    "RECEIVE_LONG":
        "Longitude associated with the transaction receiver "
        "or merchant.",

    "TARGET_OMEGA":
        "Binary target variable indicating whether the "
        "transaction is fraudulent."
}


# ============================================================
# 04. DEFINE FEATURE TRANSFORMATION MAP
# ============================================================

transformation_records = [

    {
        "PRIMARY_FEATURE":
            "NID",

        "FINAL_FEATURE":
            "NID_ALPHA",

        "TRANSFORMATION":
            "Renamed identifier"
    },

    {
        "PRIMARY_FEATURE":
            "cc_num",

        "FINAL_FEATURE":
            "TRANS_NUM_CARD_FEWF",

        "TRANSFORMATION":
            "Renamed and prepared for future FEWF"
    },

    {
        "PRIMARY_FEATURE":
            "amt",

        "FINAL_FEATURE":
            "TRANS_VALUE",

        "TRANSFORMATION":
            "Renamed numerical feature"
    },

    {
        "PRIMARY_FEATURE":
            "trans_date_trans_time",

        "FINAL_FEATURE":
            "TRANS_DAY",

        "TRANSFORMATION":
            "Day of month extracted"
    },

    {
        "PRIMARY_FEATURE":
            "trans_date_trans_time",

        "FINAL_FEATURE":
            "TRANS_WEEK_OHEWI",

        "TRANSFORMATION":
            "Weekday extracted and prepared for future OHEWI"
    },

    {
        "PRIMARY_FEATURE":
            "trans_date_trans_time",

        "FINAL_FEATURE":
            "TRANS_YEAR_BE",

        "TRANSFORMATION":
            "Year extracted and prepared for future BE"
    },

    {
        "PRIMARY_FEATURE":
            "trans_date_trans_time",

        "FINAL_FEATURE":
            "TRANS_MONTH_SIN / TRANS_MONTH_COS",

        "TRANSFORMATION":
            "Month converted to cyclical sine/cosine representation"
    },

    {
        "PRIMARY_FEATURE":
            "trans_date_trans_time",

        "FINAL_FEATURE":
            "TRANS_HOUR_SIN / TRANS_HOUR_COS",

        "TRANSFORMATION":
            "Time converted to cyclical sine/cosine representation"
    },

    {
        "PRIMARY_FEATURE":
            "first + last",

        "FINAL_FEATURE":
            "SEND_NAME_FEWF",

        "TRANSFORMATION":
            "Full name created and prepared for future FEWF"
    },

    {
        "PRIMARY_FEATURE":
            "gender",

        "FINAL_FEATURE":
            "SEND_GENDER_BE",

        "TRANSFORMATION":
            "Renamed and prepared for future BE"
    },

    {
        "PRIMARY_FEATURE":
            "dob",

        "FINAL_FEATURE":
            "SEND_AGE",

        "TRANSFORMATION":
            "Converted to decimal age using 2026-09-03"
    },

    {
        "PRIMARY_FEATURE":
            "job",

        "FINAL_FEATURE":
            "SEND_JOB_FEWF",

        "TRANSFORMATION":
            "Renamed and prepared for future FEWF"
    },

    {
        "PRIMARY_FEATURE":
            "lat",

        "FINAL_FEATURE":
            "SEND_LAT_REGISTER",

        "TRANSFORMATION":
            "Renamed geographical feature"
    },

    {
        "PRIMARY_FEATURE":
            "long",

        "FINAL_FEATURE":
            "SEND_LONG_REGISTER",

        "TRANSFORMATION":
            "Renamed geographical feature"
    },

    {
        "PRIMARY_FEATURE":
            "city_pop",

        "FINAL_FEATURE":
            "SEND_POP_REGISTER",

        "TRANSFORMATION":
            "Renamed population feature"
    },

    {
        "PRIMARY_FEATURE":
            "merchant",

        "FINAL_FEATURE":
            "RECEIVE_LOC_FEWF",

        "TRANSFORMATION":
            "Renamed and prepared for future FEWF"
    },

    {
        "PRIMARY_FEATURE":
            "category",

        "FINAL_FEATURE":
            "RECEIVE_CATEGORY_OHEWI",

        "TRANSFORMATION":
            "Renamed and prepared for future OHEWI"
    },

    {
        "PRIMARY_FEATURE":
            "merch_lat",

        "FINAL_FEATURE":
            "RECEIVE_LAT",

        "TRANSFORMATION":
            "Renamed receiver latitude"
    },

    {
        "PRIMARY_FEATURE":
            "merch_long",

        "FINAL_FEATURE":
            "RECEIVE_LONG",

        "TRANSFORMATION":
            "Renamed receiver longitude"
    },

    {
        "PRIMARY_FEATURE":
            "is_fraud / is_fraude",

        "FINAL_FEATURE":
            "TARGET_OMEGA",

        "TRANSFORMATION":
            "Renamed target variable"
    }
]


transformation_table = pd.DataFrame(
    transformation_records
)


# ============================================================
# 05. DEFINE FEATURES INTENTIONALLY REMOVED
# ============================================================

removed_features = [
    "street",
    "city",
    "state",
    "zip",
    "trans_num",
    "unix_time"
]


# ============================================================
# 06. ENSURE THE REPORT DIRECTORY EXISTS
# ============================================================

report_file.parent.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 07. RECOVER OR REMOVE STALE REPORT BACKUP
# ============================================================

if backup_report_file.exists():

    if report_file.exists():

        backup_report_file.unlink()

        print(
            "\nStale comparison-report backup removed."
        )

    else:

        shutil.move(
            str(
                backup_report_file
            ),
            str(
                report_file
            )
        )

        print(
            "\nPrevious comparison report restored from backup."
        )


# ============================================================
# 08. REMOVE STALE TEMPORARY REPORT
# ============================================================

if temporary_report_file.exists():

    temporary_report_file.unlink()

    print(
        "\nPrevious temporary comparison report removed."
    )


# ============================================================
# 09. VALIDATE REQUIRED DATASET FILES
# ============================================================

primary_file_is_valid = (
    primary_dataset_file.exists()
    and
    primary_dataset_file.is_file()
    and
    primary_dataset_file.stat().st_size > 0
)


final_file_is_valid = (
    final_dataset_file.exists()
    and
    final_dataset_file.is_file()
    and
    final_dataset_file.stat().st_size > 0
)


if not primary_file_is_valid:

    raise FileNotFoundError(
        "Primary dataset is missing or empty: "
        f"{primary_dataset_file}"
    )


if not final_file_is_valid:

    raise FileNotFoundError(
        "Final dataset is missing or empty: "
        f"{final_dataset_file}"
    )


print(
    "\nRequired datasets validated successfully."
)


# ============================================================
# 10. READ FINAL PARQUET METADATA
# ============================================================

final_parquet_metadata = pq.ParquetFile(
    final_dataset_file
)


parquet_rows = int(
    final_parquet_metadata.metadata.num_rows
)


parquet_columns = int(
    final_parquet_metadata.metadata.num_columns
)


parquet_feature_names = (
    final_parquet_metadata
    .schema_arrow
    .names
)


del final_parquet_metadata

gc.collect()


# ============================================================
# 11. LOAD THE PRIMARY DATASET
# ============================================================

print(
    "\nLoading the primary dataset..."
)


primary_df = pd.read_csv(
    primary_dataset_file,
    low_memory=False
)


if primary_df.empty:

    raise ValueError(
        "The primary dataset is empty."
    )


primary_shape = (
    primary_df.shape
)


primary_columns = (
    primary_df
    .columns
    .tolist()
)


primary_dtypes = (
    primary_df
    .dtypes
    .astype(str)
    .copy()
)


primary_types_table = pd.DataFrame({

    "FEATURE":
        primary_dtypes.index,

    "TYPE":
        primary_dtypes.values
})


primary_missing_by_feature = (
    primary_df
    .isna()
    .sum()
)


primary_missing_values = int(
    primary_missing_by_feature.sum()
)


primary_missing_table = pd.DataFrame({

    "FEATURE":
        primary_missing_by_feature.index,

    "MISSING_VALUES":
        primary_missing_by_feature.values
})


# ============================================================
# 12. IDENTIFY THE PRIMARY TARGET FEATURE
# ============================================================

primary_target_candidates = [
    feature
    for feature in [
        "is_fraud",
        "is_fraude"
    ]
    if feature in primary_df.columns
]


if len(
    primary_target_candidates
) != 1:

    raise ValueError(
        "The primary dataset must contain exactly one target "
        "feature named is_fraud or is_fraude."
    )


primary_target_feature = (
    primary_target_candidates[0]
)


# ============================================================
# 13. VALIDATE AND STORE THE PRIMARY IDENTIFIER
# ============================================================

if "NID" not in primary_df.columns:

    raise KeyError(
        "NID was not found in the primary dataset."
    )


primary_nid_numeric = pd.to_numeric(
    primary_df[
        "NID"
    ],
    errors="coerce"
)


if primary_nid_numeric.isna().any():

    raise ValueError(
        "Primary NID contains missing or non-numeric values."
    )


primary_nid_array = (
    primary_nid_numeric
    .to_numpy(
        dtype=np.int64,
        copy=True
    )
)


primary_nid_is_unique = (
    len(
        np.unique(
            primary_nid_array
        )
    )
    ==
    len(
        primary_nid_array
    )
)


primary_nid_is_sequential = np.array_equal(

    primary_nid_array,

    np.arange(
        1,
        len(
            primary_nid_array
        )
        +
        1,
        dtype=np.int64
    )
)


# ============================================================
# 14. VALIDATE AND STORE THE PRIMARY TARGET
# ============================================================

primary_target_numeric = pd.to_numeric(
    primary_df[
        primary_target_feature
    ],
    errors="coerce"
)


if primary_target_numeric.isna().any():

    raise ValueError(
        "Primary target contains missing or non-numeric values."
    )


primary_target_unique_values = set(
    primary_target_numeric.unique()
)


if not primary_target_unique_values.issubset(
    {
        0,
        1
    }
):

    raise ValueError(
        "Primary target contains values outside {0, 1}."
    )


primary_target_array = (
    primary_target_numeric
    .to_numpy(
        dtype=np.int8,
        copy=True
    )
)


primary_target_distribution = (
    primary_target_numeric
    .value_counts(
        dropna=False
    )
    .sort_index()
)


print(
    "Primary dataset loaded and validated successfully."
)


# ============================================================
# 15. RELEASE THE PRIMARY DATASET FROM MEMORY
# ============================================================

del primary_df
del primary_nid_numeric
del primary_target_numeric

gc.collect()


# ============================================================
# 16. LOAD THE FINAL DATASET
# ============================================================

print(
    "\nLoading the final dataset..."
)


final_df = pd.read_parquet(
    final_dataset_file,
    engine="pyarrow"
)


if final_df.empty:

    raise ValueError(
        "The final dataset is empty."
    )


final_shape = (
    final_df.shape
)


final_columns = (
    final_df
    .columns
    .tolist()
)


final_dtypes = (
    final_df
    .dtypes
    .astype(str)
    .copy()
)


final_types_table = pd.DataFrame({

    "FEATURE":
        final_dtypes.index,

    "TYPE":
        final_dtypes.values
})


final_missing_by_feature = (
    final_df
    .isna()
    .sum()
)


final_missing_values = int(
    final_missing_by_feature.sum()
)


final_missing_table = pd.DataFrame({

    "FEATURE":
        final_missing_by_feature.index,

    "MISSING_VALUES":
        final_missing_by_feature.values
})


print(
    "Final dataset loaded successfully."
)


# ============================================================
# 17. VALIDATE FINAL DATASET STRUCTURE
# ============================================================

final_structure_matches_expected = (
    final_columns
    ==
    expected_final_columns
)


final_feature_count_matches_expected = (
    final_shape[1]
    ==
    len(
        expected_final_columns
    )
)


parquet_structure_matches_loaded = (
    parquet_rows
    ==
    final_shape[0]

    and

    parquet_columns
    ==
    final_shape[1]

    and

    parquet_feature_names
    ==
    final_columns
)


# ============================================================
# 18. VALIDATE FINAL IDENTIFIER
# ============================================================

if "NID_ALPHA" not in final_df.columns:

    raise KeyError(
        "NID_ALPHA was not found in the final dataset."
    )


final_nid_numeric = pd.to_numeric(
    final_df[
        "NID_ALPHA"
    ],
    errors="coerce"
)


final_nid_valid = (
    not final_nid_numeric.isna().any()
)


if final_nid_valid:

    final_nid_array = (
        final_nid_numeric
        .to_numpy(
            dtype=np.int64,
            copy=True
        )
    )


    final_nid_is_unique = (
        len(
            np.unique(
                final_nid_array
            )
        )
        ==
        len(
            final_nid_array
        )
    )


    final_nid_is_sequential = np.array_equal(

        final_nid_array,

        np.arange(
            1,
            len(
                final_nid_array
            )
            +
            1,
            dtype=np.int64
        )
    )


    nid_preserved_exactly = (
        primary_nid_array.shape
        ==
        final_nid_array.shape

        and

        np.array_equal(
            primary_nid_array,
            final_nid_array
        )
    )


else:

    final_nid_is_unique = False

    final_nid_is_sequential = False

    nid_preserved_exactly = False


# ============================================================
# 19. VALIDATE FINAL TARGET
# ============================================================

if "TARGET_OMEGA" not in final_df.columns:

    raise KeyError(
        "TARGET_OMEGA was not found in the final dataset."
    )


final_target_numeric = pd.to_numeric(
    final_df[
        "TARGET_OMEGA"
    ],
    errors="coerce"
)


final_target_valid = (
    not final_target_numeric.isna().any()
)


if final_target_valid:

    final_target_unique_values = set(
        final_target_numeric.unique()
    )


    final_target_is_binary = (
        final_target_unique_values
        .issubset(
            {
                0,
                1
            }
        )
    )


    final_target_array = (
        final_target_numeric
        .to_numpy(
            dtype=np.int8,
            copy=True
        )
    )


    target_preserved_exactly = (
        primary_target_array.shape
        ==
        final_target_array.shape

        and

        np.array_equal(
            primary_target_array,
            final_target_array
        )
    )


else:

    final_target_is_binary = False

    target_preserved_exactly = False


final_target_distribution = (
    final_target_numeric
    .value_counts(
        dropna=False
    )
    .sort_index()
)


target_distribution_preserved = (
    primary_target_distribution.equals(
        final_target_distribution
    )
)


# ============================================================
# 20. VALIDATE OBSERVATION COUNT
# ============================================================

row_count_preserved = (
    primary_shape[0]
    ==
    final_shape[0]
)


# ============================================================
# 21. VALIDATE FINAL MISSING VALUES
# ============================================================

final_has_no_missing_values = (
    final_missing_values
    ==
    0
)


# ============================================================
# 22. CALCULATE FILE SIZES
# ============================================================

primary_size_bytes = int(
    primary_dataset_file
    .stat()
    .st_size
)


final_size_bytes = int(
    final_dataset_file
    .stat()
    .st_size
)


primary_size_mb = (
    primary_size_bytes
    /
    1024**2
)


final_size_mb = (
    final_size_bytes
    /
    1024**2
)


space_saved_bytes = (
    primary_size_bytes
    -
    final_size_bytes
)


space_saved_mb = (
    space_saved_bytes
    /
    1024**2
)


space_reduction_percentage = (

    1

    -

    (
        final_size_bytes
        /
        primary_size_bytes
    )

) * 100


# ============================================================
# 23. IDENTIFY RAW FEATURE-NAME DIFFERENCES
# ============================================================

primary_only_features = [
    column
    for column in primary_columns
    if column not in final_columns
]


final_only_features = [
    column
    for column in final_columns
    if column not in primary_columns
]


# ============================================================
# 24. CREATE THE FINAL FEATURE SUMMARY
# ============================================================

feature_summary = pd.DataFrame({

    "FEATURE":
        final_columns,

    "TYPE":
        [
            str(
                final_df[
                    column
                ].dtype
            )
            for column in final_columns
        ],

    "UNIQUE_VALUES":
        [
            int(
                final_df[
                    column
                ]
                .nunique(
                    dropna=False
                )
            )
            for column in final_columns
        ],

    "MISSING_VALUES":
        [
            int(
                final_df[
                    column
                ]
                .isna()
                .sum()
            )
            for column in final_columns
        ],

    "DESCRIPTION":
        [
            feature_descriptions.get(
                column,
                "Description not defined."
            )
            for column in final_columns
        ]
})


# ============================================================
# 25. CREATE THE INTEGRITY CHECK TABLE
# ============================================================

integrity_records = [

    {
        "CHECK":
            "Primary NID is unique",

        "STATUS":
            primary_nid_is_unique
    },

    {
        "CHECK":
            "Primary NID is sequential",

        "STATUS":
            primary_nid_is_sequential
    },

    {
        "CHECK":
            "Primary and final row counts are identical",

        "STATUS":
            row_count_preserved
    },

    {
        "CHECK":
            "Final feature count is exactly 22",

        "STATUS":
            final_feature_count_matches_expected
    },

    {
        "CHECK":
            "Final feature names and order match expected schema",

        "STATUS":
            final_structure_matches_expected
    },

    {
        "CHECK":
            "Parquet metadata matches loaded final dataset",

        "STATUS":
            parquet_structure_matches_loaded
    },

    {
        "CHECK":
            "Final NID_ALPHA is valid",

        "STATUS":
            final_nid_valid
    },

    {
        "CHECK":
            "Final NID_ALPHA is unique",

        "STATUS":
            final_nid_is_unique
    },

    {
        "CHECK":
            "Final NID_ALPHA is sequential",

        "STATUS":
            final_nid_is_sequential
    },

    {
        "CHECK":
            "NID was preserved exactly and in the same order",

        "STATUS":
            nid_preserved_exactly
    },

    {
        "CHECK":
            "Final TARGET_OMEGA is valid",

        "STATUS":
            final_target_valid
    },

    {
        "CHECK":
            "Final TARGET_OMEGA is binary",

        "STATUS":
            final_target_is_binary
    },

    {
        "CHECK":
            "TARGET values were preserved exactly row by row",

        "STATUS":
            target_preserved_exactly
    },

    {
        "CHECK":
            "TARGET distribution was preserved",

        "STATUS":
            target_distribution_preserved
    },

    {
        "CHECK":
            "Final dataset contains no missing values",

        "STATUS":
            final_has_no_missing_values
    }
]


integrity_table = pd.DataFrame(
    integrity_records
)


integrity_table[
    "RESULT"
] = np.where(
    integrity_table[
        "STATUS"
    ],
    "PASS",
    "FAIL"
)


all_integrity_checks_passed = bool(
    integrity_table[
        "STATUS"
    ]
    .all()
)


# ============================================================
# 26. CREATE TARGET DISTRIBUTION COMPARISON
# ============================================================

target_distribution_table = pd.DataFrame({

    "PRIMARY":
        primary_target_distribution,

    "FINAL":
        final_target_distribution
})


target_distribution_table = (
    target_distribution_table
    .fillna(
        0
    )
)


target_distribution_table.index.name = (
    "TARGET_VALUE"
)


# ============================================================
# 27. CREATE A REPRODUCIBLE RANDOM SAMPLE
# ============================================================

actual_sample_size = min(
    random_sample_size,
    len(
        final_df
    )
)


random_sample = (
    final_df
    .sample(
        n=actual_sample_size,
        random_state=random_state
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 28. CREATE REPORT HEADER
# ============================================================

report_lines = []


report_lines.append(
    "=" * 100
)

report_lines.append(
    "PRIMARY-TO-FINAL DATASET INTEGRITY VALIDATION "
    "AND COMPARISON REPORT"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    "\nOverall preprocessing integrity status: "
    + (
        "PASS"
        if all_integrity_checks_passed
        else "FAIL"
    )
)


# ============================================================
# 29. ADD INTEGRITY CHECKS TO REPORT
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "PREPROCESSING INTEGRITY CHECKS"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    integrity_table[
        [
            "CHECK",
            "RESULT"
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 30. ADD GENERAL DATASET COMPARISON
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "GENERAL DATASET COMPARISON"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    f"\nPrimary dataset rows: "
    f"{primary_shape[0]}"
)

report_lines.append(
    f"Final dataset rows:   "
    f"{final_shape[0]}"
)

report_lines.append(
    f"\nPrimary dataset features: "
    f"{primary_shape[1]}"
)

report_lines.append(
    f"Final dataset features:   "
    f"{final_shape[1]}"
)

report_lines.append(
    f"\nPrimary dataset missing values: "
    f"{primary_missing_values}"
)

report_lines.append(
    f"Final dataset missing values:   "
    f"{final_missing_values}"
)

report_lines.append(
    f"\nParquet metadata rows:     "
    f"{parquet_rows}"
)

report_lines.append(
    f"Parquet metadata features: "
    f"{parquet_columns}"
)


# ============================================================
# 31. ADD DISK SPACE COMPARISON
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "DISK SPACE COMPARISON"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    f"\nPrimary CSV size:  "
    f"{primary_size_mb:.2f} MB"
)

report_lines.append(
    f"Final Parquet size: "
    f"{final_size_mb:.2f} MB"
)

report_lines.append(
    f"Space saved:        "
    f"{space_saved_mb:.2f} MB"
)

report_lines.append(
    f"Space reduction:    "
    f"{space_reduction_percentage:.2f}%"
)


# ============================================================
# 32. ADD TARGET DISTRIBUTION COMPARISON
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "TARGET DISTRIBUTION COMPARISON"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    target_distribution_table.to_string()
)


report_lines.append(
    "\nExact row-by-row target preservation: "
    + (
        "PASS"
        if target_preserved_exactly
        else "FAIL"
    )
)


# ============================================================
# 33. ADD PRIMARY DATA TYPES
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "DATA TYPES - PRIMARY DATASET"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    primary_types_table.to_string(
        index=False
    )
)


# ============================================================
# 34. ADD FINAL DATA TYPES
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "DATA TYPES - FINAL DATASET"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    final_types_table.to_string(
        index=False
    )
)


# ============================================================
# 35. ADD FEATURE TRANSFORMATION MAP
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "PRIMARY-TO-FINAL FEATURE TRANSFORMATION MAP"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    transformation_table.to_string(
        index=False
    )
)


# ============================================================
# 36. ADD INTENTIONALLY REMOVED FEATURES
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "FEATURES INTENTIONALLY REMOVED DURING PREPROCESSING"
)

report_lines.append(
    "=" * 100
)


for feature in removed_features:

    report_lines.append(
        f"- {feature}"
    )


# ============================================================
# 37. ADD RAW FEATURE-NAME DIFFERENCES
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "RAW FEATURE-NAME DIFFERENCES"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    "\nFeatures present by name only in the primary dataset:"
)


if primary_only_features:

    for feature in primary_only_features:

        report_lines.append(
            f"- {feature}"
        )

else:

    report_lines.append(
        "No exclusive features."
    )


report_lines.append(
    "\nFeatures present by name only in the final dataset:"
)


if final_only_features:

    for feature in final_only_features:

        report_lines.append(
            f"- {feature}"
        )

else:

    report_lines.append(
        "No exclusive features."
    )


report_lines.append(
    "\nNote: these name differences include expected renaming, "
    "feature engineering, and feature removal. They should not "
    "be interpreted as data loss by themselves."
)


# ============================================================
# 38. ADD PRIMARY MISSING-VALUE SUMMARY
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "MISSING VALUES - PRIMARY DATASET"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    primary_missing_table.to_string(
        index=False
    )
)


# ============================================================
# 39. ADD FINAL MISSING-VALUE SUMMARY
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "MISSING VALUES - FINAL DATASET"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    final_missing_table.to_string(
        index=False
    )
)


# ============================================================
# 40. ADD FINAL FEATURE SUMMARY
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "FINAL FEATURE SUMMARY"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    feature_summary.to_string(
        index=False
    )
)


# ============================================================
# 41. ADD RANDOM SAMPLE
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    f"RANDOM SAMPLE - {actual_sample_size} ROWS"
)

report_lines.append(
    "=" * 100
)


report_lines.append(
    random_sample.to_string(
        index=False
    )
)


# ============================================================
# 42. ADD FINAL CONCLUSION
# ============================================================

report_lines.append(
    "\n\n"
    + "=" * 100
)

report_lines.append(
    "FINAL PREPROCESSING VALIDATION RESULT"
)

report_lines.append(
    "=" * 100
)


if all_integrity_checks_passed:

    report_lines.append(
        "\nPASS"
    )

    report_lines.append(
        "\nAll defined preprocessing integrity checks passed."
    )

    report_lines.append(
        "The final dataset preserved the expected observations, "
        "identifier order, target values, feature structure, "
        "and missing-value requirements."
    )

else:

    report_lines.append(
        "\nFAIL"
    )

    report_lines.append(
        "\nOne or more preprocessing integrity checks failed."
    )

    report_lines.append(
        "The failed checks must be reviewed before the final "
        "dataset is considered approved for subsequent analysis."
    )


# ============================================================
# 43. BUILD THE COMPLETE REPORT
# ============================================================

report_text = "\n".join(
    report_lines
)


# ============================================================
# 44. SAVE THE REPORT TO A TEMPORARY FILE
# ============================================================

temporary_report_file.write_text(
    report_text,
    encoding="utf-8"
)


temporary_report_is_valid = (
    temporary_report_file.exists()
    and
    temporary_report_file.is_file()
    and
    temporary_report_file.stat().st_size > 0
)


if not temporary_report_is_valid:

    raise FileNotFoundError(
        "The temporary comparison report was not "
        "created correctly."
    )


# ============================================================
# 45. SAFELY REPLACE THE PREVIOUS REPORT
# ============================================================

report_already_existed = (
    report_file.exists()
)


if backup_report_file.exists():

    backup_report_file.unlink()


if report_already_existed:

    shutil.move(
        str(
            report_file
        ),
        str(
            backup_report_file
        )
    )


try:

    shutil.move(
        str(
            temporary_report_file
        ),
        str(
            report_file
        )
    )


    report_is_valid = (
        report_file.exists()
        and
        report_file.is_file()
        and
        report_file.stat().st_size > 0
    )


    if not report_is_valid:

        raise FileNotFoundError(
            "dataset_comparison_report.txt was not "
            "created correctly."
        )


except Exception:

    if report_file.exists():

        report_file.unlink()


    if backup_report_file.exists():

        shutil.move(
            str(
                backup_report_file
            ),
            str(
                report_file
            )
        )


        print(
            "\nPrevious comparison report restored after "
            "replacement failure."
        )


    raise


# ============================================================
# 46. REMOVE THE REPORT BACKUP AFTER SUCCESS
# ============================================================

if backup_report_file.exists():

    backup_report_file.unlink()


# ============================================================
# 47. DISPLAY REPORT CREATION STATUS
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "COMPARISON REPORT"
)

print(
    "=" * 100
)


if report_already_existed:

    print(
        "Previous dataset_comparison_report.txt "
        "overwritten successfully."
    )

else:

    print(
        "dataset_comparison_report.txt "
        "created successfully."
    )


print(
    "Location:",
    report_file
)


print(
    "File size:",
    f"{report_file.stat().st_size / 1024**2:.4f} MB"
)


# ============================================================
# 48. DISPLAY INTEGRITY CHECKS
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "PREPROCESSING INTEGRITY CHECKS"
)

print(
    "=" * 100
)


display(
    integrity_table[
        [
            "CHECK",
            "RESULT"
        ]
    ]
)


# ============================================================
# 49. DISPLAY GENERAL COMPARISON
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "GENERAL DATASET COMPARISON"
)

print(
    "=" * 100
)


print(
    "Primary dataset rows:",
    primary_shape[0]
)

print(
    "Final dataset rows:  ",
    final_shape[0]
)


print(
    "\nPrimary dataset features:",
    primary_shape[1]
)

print(
    "Final dataset features:  ",
    final_shape[1]
)


print(
    "\nPrimary dataset missing values:",
    primary_missing_values
)

print(
    "Final dataset missing values:  ",
    final_missing_values
)


# ============================================================
# 50. DISPLAY DISK SPACE COMPARISON
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "DISK SPACE COMPARISON"
)

print(
    "=" * 100
)


print(
    "Primary CSV:",
    f"{primary_size_mb:.2f} MB"
)

print(
    "Final Parquet:",
    f"{final_size_mb:.2f} MB"
)

print(
    "Space saved:",
    f"{space_saved_mb:.2f} MB"
)

print(
    "Space reduction:",
    f"{space_reduction_percentage:.2f}%"
)


# ============================================================
# 51. DISPLAY TARGET DISTRIBUTION COMPARISON
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "TARGET DISTRIBUTION COMPARISON"
)

print(
    "=" * 100
)


display(
    target_distribution_table
)


print(
    "\nExact row-by-row target preservation:",
    (
        "PASS"
        if target_preserved_exactly
        else "FAIL"
    )
)


# ============================================================
# 52. DISPLAY DATA TYPES
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "DATA TYPES - PRIMARY DATASET"
)

print(
    "=" * 100
)


display(
    primary_types_table
)


print(
    "\n"
    + "=" * 100
)

print(
    "DATA TYPES - FINAL DATASET"
)

print(
    "=" * 100
)


display(
    final_types_table
)


# ============================================================
# 53. DISPLAY FEATURE TRANSFORMATION MAP
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "PRIMARY-TO-FINAL FEATURE TRANSFORMATION MAP"
)

print(
    "=" * 100
)


display(
    transformation_table
)


# ============================================================
# 54. DISPLAY FINAL FEATURE SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL FEATURE SUMMARY"
)

print(
    "=" * 100
)


display(
    feature_summary
)


# ============================================================
# 55. DISPLAY RANDOM SAMPLE
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    f"RANDOM SAMPLE - {actual_sample_size} ROWS"
)

print(
    "=" * 100
)


display(
    random_sample
)


# ============================================================
# 56. DISPLAY FINAL VALIDATION RESULT
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "FINAL PREPROCESSING VALIDATION RESULT"
)

print(
    "=" * 100
)


if all_integrity_checks_passed:

    print(
        "PASS"
    )

    print(
        "All defined preprocessing integrity checks passed."
    )

else:

    print(
        "FAIL"
    )

    print(
        "One or more preprocessing integrity checks failed."
    )


# ============================================================
# 57. RELEASE MEMORY
# ============================================================

del final_df

del primary_nid_array
del primary_target_array

if "final_nid_array" in globals():

    del final_nid_array

if "final_target_array" in globals():

    del final_target_array

gc.collect()


print(
    "\nComparison objects released from memory."
)


# ============================================================
# 58. STOP THE PIPELINE IF AN INTEGRITY CHECK FAILED
# ============================================================

if not all_integrity_checks_passed:

    failed_checks = (
        integrity_table.loc[
            ~integrity_table[
                "STATUS"
            ],
            "CHECK"
        ]
        .tolist()
    )


    raise ValueError(
        "Preprocessing integrity validation failed. "
        "Failed checks: "
        + "; ".join(
            failed_checks
        )
    )


print(
    "\nPreprocessing comparison completed successfully."
)


Required datasets validated successfully.

Loading the primary dataset...
Primary dataset loaded and validated successfully.

Loading the final dataset...
Final dataset loaded successfully.

COMPARISON REPORT
Previous dataset_comparison_report.txt overwritten successfully.
Location: /projeto_tcc_2026/data/final_dataset/dataset_comparison_report.txt
File size: 0.0199 MB

PREPROCESSING INTEGRITY CHECKS


,CHECK,RESULT
0,Primary NID is unique,PASS
1,Primary NID is sequential,PASS
2,Primary and final row counts are identical,PASS
3,Final feature count is exactly 22,PASS
4,Final feature names and order match expected s...,PASS
5,Parquet metadata matches loaded final dataset,PASS
6,Final NID_ALPHA is valid,PASS
7,Final NID_ALPHA is unique,PASS
8,Final NID_ALPHA is sequential,PASS
9,NID was preserved exactly and in the same order,PASS



GENERAL DATASET COMPARISON
Primary dataset rows: 1852394
Final dataset rows:   1852394

Primary dataset features: 23
Final dataset features:   22

Primary dataset missing values: 0
Final dataset missing values:   0

DISK SPACE COMPARISON
Primary CSV: 472.78 MB
Final Parquet: 64.34 MB
Space saved: 408.44 MB
Space reduction: 86.39%

TARGET DISTRIBUTION COMPARISON


,PRIMARY,FINAL
TARGET_VALUE,,
0,1842743,1842743
1,9651,9651



Exact row-by-row target preservation: PASS

DATA TYPES - PRIMARY DATASET


...


FINAL FEATURE SUMMARY


,FEATURE,TYPE,UNIQUE_VALUES,MISSING_VALUES,DESCRIPTION
0,NID_ALPHA,int32,1852394,0,Unique sequential identifier assigned to each ...
1,TRANS_NUM_CARD_FEWF,int64,999,0,Credit card number feature preserved in its or...
2,TRANS_VALUE,float64,60616,0,Monetary value of the transaction.
3,TRANS_DAY,int8,31,0,Day of the month on which the transaction occu...
4,TRANS_WEEK_OHEWI,category,7,0,Day-of-week feature preserved as the original ...
5,TRANS_YEAR_BE,int16,2,0,Transaction year preserved as the original yea...
6,TRANS_MONTH_SIN,float64,11,0,Sine component of the cyclical representation ...
7,TRANS_MONTH_COS,float64,11,0,Cosine component of the cyclical representatio...
8,TRANS_HOUR_SIN,float64,73812,0,Sine component of the cyclical representation ...
9,TRANS_HOUR_COS,float64,75789,0,Cosine component of the cyclical representatio...



RANDOM SAMPLE - 10 ROWS


,NID_ALPHA,TRANS_NUM_CARD_FEWF,TRANS_VALUE,TRANS_DAY,TRANS_WEEK_OHEWI,TRANS_YEAR_BE,TRANS_MONTH_SIN,TRANS_MONTH_COS,TRANS_HOUR_SIN,TRANS_HOUR_COS,...,SEND_AGE,SEND_JOB_FEWF,SEND_LAT_REGISTER,SEND_LONG_REGISTER,SEND_POP_REGISTER,RECEIVE_LOC_FEWF,RECEIVE_CATEGORY_OHEWI,RECEIVE_LAT,RECEIVE_LONG,TARGET_OMEGA
0,1541145,5359543825610251,59.91,18,Friday,2020,-8.660254e-01,-5.000000e-01,0.948807,-0.315856,...,51.180000,"Engineer, drilling",45.780102,-111.143898,18182,"fraud_Jenkins, Hauck and Friesen",gas_transport,45.274075,-111.649432,0
1,1731582,5540636818935089,3.96,5,Saturday,2020,-5.000000e-01,8.660254e-01,-0.998723,-0.050520,...,41.419998,Geoscientist,42.691101,-71.160500,76383,fraud_Jast-McDermott,shopping_pos,43.356278,-71.008959,0
2,354660,2720894374956739,51.17,15,Saturday,2019,5.000000e-01,-8.660254e-01,0.153273,-0.988184,...,99.279999,"Psychologist, sport and exercise",42.597801,-82.882301,16305,fraud_Bartoletti-Wunsch,gas_transport,42.372483,-83.508020,0
3,1493789,6011438889172900,2.06,29,Saturday,2020,-5.000000e-01,-8.660254e-01,-0.298971,0.954262,...,33.410000,Electrical engineer,34.285301,-91.333603,5161,"fraud_Roob, Conn and Tremblay",shopping_pos,33.833389,-91.158293,0
4,468149,60495593109,6.58,25,Thursday,2019,1.224647e-16,-1.000000e+00,-0.844756,-0.535151,...,83.779999,Television camera operator,32.769901,-96.742996,1263321,"fraud_Kilback, Nitzsche and Leffler",travel,32.458643,-96.577001,0
5,1389855,6011477612335392,24.49,23,Thursday,2020,1.224647e-16,-1.000000e+00,-0.991340,-0.131319,...,82.709999,"Designer, television/film set",40.406200,-84.507599,2274,"fraud_Cormier, Stracke and Thiel",entertainment,39.503889,-84.030238,0
6,1760907,2703186189652095,99.74,11,Friday,2020,-5.000000e-01,8.660254e-01,-0.229271,0.973363,...,38.490002,"Psychologist, counselling",36.078800,-81.178101,3495,fraud_Skiles LLC,home,36.821934,-81.063288,0
7,469448,4839615922685395,147.12,26,Friday,2019,1.224647e-16,-1.000000e+00,0.962139,0.272560,...,71.330002,Social researcher,39.013000,-86.545700,76,fraud_Vandervort-Funk,grocery_pos,38.202346,-86.136019,0
8,1835019,3553629419254918,46.85,28,Monday,2020,-5.000000e-01,8.660254e-01,0.911881,-0.410454,...,42.000000,"Research officer, political party",48.340000,-122.345596,85,fraud_Sporer Inc,gas_transport,47.631938,-122.506513,0
9,240728,4481131401752,33.56,30,Tuesday,2019,1.000000e+00,6.123234e-17,-0.731849,-0.681466,...,51.349998,English as a second language teacher,42.284801,-71.720497,35299,"fraud_Wintheiser, Dietrich and Schimmel",misc_pos,42.740677,-72.593286,0



FINAL PREPROCESSING VALIDATION RESULT
PASS
All defined preprocessing integrity checks passed.

Comparison objects released from memory.

Preprocessing comparison completed successfully.
